<a href="https://colab.research.google.com/github/UTD2026/Mixed_Dataset_Testing_STA/blob/main/TLM_Upgrade_Tests_Masking_Dynamic_Forgetting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TLM TTL — Base Qwen vs CE vs Gradient Selection + AAAI Upgrade Tests

Includes the aligned-reference baseline workflow plus extra experiment cells for:
- punctuation/whitespace token masking,
- rolling dynamic percentile thresholds,
- combined masking + dynamic threshold variants,
- catastrophic forgetting / general-language check,
- end diagnostics to rank the new variants.


## 0 — Clone / enter repo

In [1]:
from pathlib import Path
import os

if not Path('/content/TLM').exists():
    !git clone https://github.com/Fhujinwu/TLM.git /content/TLM

%cd /content/TLM
print('Repo root:', os.getcwd())
!git rev-parse --short HEAD

Cloning into '/content/TLM'...
remote: Enumerating objects: 429, done.
remote: Counting objects: 100% (429/429), done.
remote: Compressing objects: 100% (320/320), done.
remote: Total 429 (delta 139), reused 354 (delta 95), pack-reused 0 (from 0)
Receiving objects: 100% (429/429), 24.73 MiB | 26.24 MiB/s, done.
Resolving deltas: 100% (139/139), done.
/content/TLM
Repo root: /content/TLM
496c9ed


## 1 — Install dependencies

In [2]:
# Install-only cell. Restart runtime after this cell.
%cd /content/TLM

!pip -q uninstall -y transformers tokenizers trl accelerate peft datasets

!pip -q install --no-cache-dir   "transformers==4.46.1"   "tokenizers>=0.20,<0.21"   "datasets==3.1.0"   "accelerate==0.34.2"   "peft==0.12.0"   "trl==0.9.6"

!pip -q install --no-cache-dir   "numpy==1.26.4"   "scipy==1.13.1"   "scikit-learn==1.5.2"   "pandas==2.2.2"   nltk rouge-score bert-score sentencepiece protobuf einops tiktoken huggingface_hub

print('✅ Dependencies installed. NOW RESTART RUNTIME before continuing.')

/content/TLM
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 76.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 136.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 477.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 903.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 244.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.8/245.8 kB 696.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 287.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 797.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 496.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 185.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 840.2 MB/s eta 0:00:00
ERROR: pip's dependen

## 1.5 — Import check after restart

In [1]:
import numpy as np
import nltk
import transformers, tokenizers, datasets, accelerate, peft

nltk.download('punkt')
nltk.download('punkt_tab')

print('numpy:', np.__version__)
print('transformers:', transformers.__version__)
print('tokenizers:', tokenizers.__version__)
print('datasets:', datasets.__version__)
print('accelerate:', accelerate.__version__)
print('peft:', peft.__version__)
print('✅ Imports are clean.')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


numpy: 1.26.4
transformers: 4.46.1
tokenizers: 0.20.3
datasets: 3.1.0
accelerate: 0.34.2
peft: 0.12.0
✅ Imports are clean.


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


## 2 — Install rouge-chinese before training/comparison

In [2]:
%cd /content/TLM
!pip -q install --no-cache-dir rouge-chinese jieba
print('✅ rouge-chinese installed.')

/content/TLM
✅ rouge-chinese installed.


## 3 — Global random-sample / seed / score settings

In [64]:
DATA_SEED = 2030
TRAIN_SEED = 2030
SAMPLE_SIZE = 500
MAX_NEW_TOKENS = 32

# 75% matched-selection static thresholds from the calibration run.
# Change these if you intentionally run a 50%/25% budget ablation.
OFFICIAL_CE_THRESHOLD = 3.537274
GRAD_NORM_THRESHOLD = 0.587426
INFLUENCE_COS_THRESHOLD = 0.734210

CUSTOM_SCORE_EXP_CLIP = 8.0
INFLUENCE_EMA_DECAY = 0.95
LOGIT_GRAD_TOP_FRAC = 1.00
GRADIENT_CHECKPOINTING = False

# AAAI upgrade knobs.
# Token mask modes: "none" or "punct_ws"
TOKEN_MASK_MODE = "none"

# Dynamic threshold modes: "none" or "rolling_percentile"
# rolling_percentile uses the score history observed so far.
DYNAMIC_THRESHOLD_MODE = "none"
DYNAMIC_THRESHOLD_PERCENTILE = 25.0  # p25 ~= top 75% selected
DYNAMIC_THRESHOLD_MIN_HISTORY = 16
DYNAMIC_THRESHOLD_MAX_HISTORY = 2048

print('DATA_SEED:', DATA_SEED)
print('TRAIN_SEED:', TRAIN_SEED)
print('SAMPLE_SIZE:', SAMPLE_SIZE)
print('MAX_NEW_TOKENS:', MAX_NEW_TOKENS)
print('OFFICIAL_CE_THRESHOLD:', OFFICIAL_CE_THRESHOLD)
print('GRAD_NORM_THRESHOLD:', GRAD_NORM_THRESHOLD)
print('INFLUENCE_COS_THRESHOLD:', INFLUENCE_COS_THRESHOLD)
print('LOGIT_GRAD_TOP_FRAC:', LOGIT_GRAD_TOP_FRAC)
print('TOKEN_MASK_MODE:', TOKEN_MASK_MODE)
print('DYNAMIC_THRESHOLD_MODE:', DYNAMIC_THRESHOLD_MODE)
print('DYNAMIC_THRESHOLD_PERCENTILE:', DYNAMIC_THRESHOLD_PERCENTILE)


DATA_SEED: 2030
TRAIN_SEED: 2030
SAMPLE_SIZE: 500
MAX_NEW_TOKENS: 32
OFFICIAL_CE_THRESHOLD: 3.537274
GRAD_NORM_THRESHOLD: 0.587426
INFLUENCE_COS_THRESHOLD: 0.73421
LOGIT_GRAD_TOP_FRAC: 1.0
TOKEN_MASK_MODE: none
DYNAMIC_THRESHOLD_MODE: none
DYNAMIC_THRESHOLD_PERCENTILE: 25.0


## 4 — Model setup

In [65]:
import os, shutil
from pathlib import Path

%cd /content/TLM
TARGET = Path('/content/TLM/models/Qwen1.5-0.5B')
TARGET.parent.mkdir(parents=True, exist_ok=True)

candidate_dirs = [
    Path('/content/TLM/models/Qwen1.5-0.5B'),
    Path('/content/models/Qwen1.5-0.5B'),
    Path('/content/Qwen1.5-0.5B'),
    Path('/content/drive/MyDrive/models/Qwen1.5-0.5B'),
    Path('/content/drive/MyDrive/Qwen1.5-0.5B'),
]

found = None
for p in candidate_dirs:
    if (p / 'config.json').exists():
        found = p
        break

if found is not None and found.resolve() != TARGET.resolve():
    print('Found existing model at:', found)
    if TARGET.exists():
        shutil.rmtree(TARGET)
    shutil.copytree(found, TARGET)
elif (TARGET / 'config.json').exists():
    print('Model already exists at:', TARGET)
else:
    print('Model not found locally. Downloading Qwen/Qwen1.5-0.5B...')
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id='Qwen/Qwen1.5-0.5B',
        local_dir=str(TARGET),
        local_dir_use_symlinks=False,
        resume_download=True,
    )

if not (TARGET / 'config.json').exists():
    raise FileNotFoundError(f'Missing config.json at {TARGET}')

print('✅ Model ready at:', TARGET)
!ls -lh /content/TLM/models/Qwen1.5-0.5B | head

/content/TLM
Model already exists at: /content/TLM/models/Qwen1.5-0.5B
✅ Model ready at: /content/TLM/models/Qwen1.5-0.5B
total 1.2G
-rw-r--r-- 1 root root  661 Jun 25 05:39 config.json
-rw-r--r-- 1 root root  138 Jun 25 05:39 generation_config.json
-rw-r--r-- 1 root root 7.2K Jun 25 05:39 LICENSE
-rw-r--r-- 1 root root 1.6M Jun 25 05:39 merges.txt
-rw-r--r-- 1 root root 1.2G Jun 25 05:39 model.safetensors
-rw-r--r-- 1 root root 2.8K Jun 25 05:39 README.md
-rw-r--r-- 1 root root 1.3K Jun 25 05:39 tokenizer_config.json
-rw-r--r-- 1 root root 6.8M Jun 25 05:39 tokenizer.json
-rw-r--r-- 1 root root 2.7M Jun 25 05:39 vocab.json


## 5 — Create randomized agriculture sample dataset

In [66]:
import json, random, hashlib
from pathlib import Path

%cd /content/TLM
DATA_ROOT = Path('/content/TLM/data')
DATASET_INFO_PATH = DATA_ROOT / 'dataset_info.json'

dataset_info = json.loads(DATASET_INFO_PATH.read_text(encoding='utf-8'))
SOURCE_KEY = 'agriculture_5k'
if SOURCE_KEY not in dataset_info:
    agri_keys = [k for k in dataset_info.keys() if 'agriculture' in k.lower()]
    print('Agriculture-like keys:', agri_keys)
    if not agri_keys:
        raise KeyError('Could not find agriculture dataset key')
    SOURCE_KEY = agri_keys[0]

source_entry = dict(dataset_info[SOURCE_KEY])
source_file = source_entry.get('file_name') or source_entry.get('file')
if source_file is None:
    raise KeyError(f'Dataset entry has no file_name/file field: {source_entry}')
source_path = DATA_ROOT / source_file
if not source_path.exists():
    raise FileNotFoundError(source_path)

raw = source_path.read_text(encoding='utf-8').strip()
if raw.startswith('['):
    full_data = json.loads(raw)
else:
    full_data = [json.loads(line) for line in raw.splitlines() if line.strip()]

if SAMPLE_SIZE > len(full_data):
    raise ValueError(f'SAMPLE_SIZE={SAMPLE_SIZE} exceeds full dataset size={len(full_data)}')

rng = random.Random(DATA_SEED)
sample_indices = sorted(rng.sample(range(len(full_data)), SAMPLE_SIZE))
sampled_data = [full_data[i] for i in sample_indices]

sample_key = f'agriculture_{SAMPLE_SIZE}_seed{DATA_SEED}'
sample_filename = f'AdaptEval/agriculture_random_sample_{SAMPLE_SIZE}_seed{DATA_SEED}.json'
sample_path = DATA_ROOT / sample_filename
sample_path.parent.mkdir(parents=True, exist_ok=True)
sample_path.write_text(json.dumps(sampled_data, ensure_ascii=False, indent=2), encoding='utf-8')

sample_entry = dict(source_entry)
if 'file_name' in sample_entry:
    sample_entry['file_name'] = sample_filename
elif 'file' in sample_entry:
    sample_entry['file'] = sample_filename
else:
    sample_entry['file_name'] = sample_filename

dataset_info[sample_key] = sample_entry
DATASET_INFO_PATH.write_text(json.dumps(dataset_info, ensure_ascii=False, indent=2), encoding='utf-8')

manifest_path = DATA_ROOT / f'AdaptEval/agriculture_random_sample_{SAMPLE_SIZE}_seed{DATA_SEED}_indices.json'
manifest = {
    'source_key': SOURCE_KEY,
    'source_file': str(source_file),
    'sample_key': sample_key,
    'sample_file': sample_filename,
    'sample_size': SAMPLE_SIZE,
    'data_seed': DATA_SEED,
    'indices': sample_indices,
}
manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')

print('✅ Created sample dataset')
print('sample_key:', sample_key)
print('sample_path:', sample_path)
print('manifest:', manifest_path)
print('sample_sha256:', hashlib.sha256(sample_path.read_bytes()).hexdigest())
print('first indices:', sample_indices[:5])
print('first row:', json.dumps(sampled_data[0], ensure_ascii=False, indent=2)[:1200])

/content/TLM
✅ Created sample dataset
sample_key: agriculture_500_seed2030
sample_path: /content/TLM/data/AdaptEval/agriculture_random_sample_500_seed2030.json
manifest: /content/TLM/data/AdaptEval/agriculture_random_sample_500_seed2030_indices.json
sample_sha256: e65e65cbc327361f3467c427394c67e26f962bd5d96d20d2c6df5efd0820bd69
first indices: [6, 7, 12, 26, 49]
first row: {
  "question": "name one difference between  fresh and dry maize in  terms of the nutrients value.",
  "answers": "Fresh maize is higher in vitamins and minerals, such as vitamin C, folate, and potassium, compared to dry maize. However, dry maize is a good source of dietary fiber, protein, and complex carbohydrates."
}


## 6 — Path sanity checks

In [67]:
from pathlib import Path
import json

%cd /content/TLM
MODEL_PATH = Path('/content/TLM/models/Qwen1.5-0.5B')
TRAINER_PATH = Path('/content/TLM/src/llamafactory/train/ttl/trainer.py')
DATASET_INFO = Path('/content/TLM/data/dataset_info.json')
sample_key = f'agriculture_{SAMPLE_SIZE}_seed{DATA_SEED}'

print('config.json exists:', (MODEL_PATH / 'config.json').exists())
print('trainer.py exists:', TRAINER_PATH.exists())
info = json.loads(DATASET_INFO.read_text(encoding='utf-8'))
print('sample key exists:', sample_key in info)
if sample_key not in info:
    raise KeyError(sample_key)

sample_file = info[sample_key].get('file_name') or info[sample_key].get('file')
print('sample_file:', sample_file)
print('sample file exists:', (Path('/content/TLM/data') / sample_file).exists())

/content/TLM
config.json exists: True
trainer.py exists: True
sample key exists: True
sample_file: AdaptEval/agriculture_random_sample_500_seed2030.json
sample file exists: True


## 7 — Official current CE TTL baseline

In [68]:
RUN_OFFICIAL_BASELINE = True

if RUN_OFFICIAL_BASELINE:
    import os, torch
    from pathlib import Path

    %cd /content/TLM
    SAMPLE_KEY = f'agriculture_{SAMPLE_SIZE}_seed{DATA_SEED}'
    OFFICIAL_OUT_NAME = (
        f"OFFICIAL_current_ce_thresh{str(OFFICIAL_CE_THRESHOLD).replace('.', 'p')}"
        f"_max{MAX_NEW_TOKENS}_sample{SAMPLE_SIZE}_seed{DATA_SEED}"
    )

    !git checkout -- src/llamafactory/train/ttl/trainer.py
    torch.cuda.empty_cache()
    os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
    os.environ['TLM_TTL_SCORE_METHOD'] = 'raw_ce'
    os.environ['TLM_TOKEN_MASK_MODE'] = 'none'
    os.environ['TLM_DYNAMIC_THRESHOLD_MODE'] = 'none'

    official_config = f'''
### model
model_name_or_path: models/Qwen1.5-0.5B

### method
stage: ttl
do_train: true
do_predict: true
finetuning_type: lora
lora_target: q_proj,v_proj
trust_remote_code: true
lora_rank: 4
lora_dropout: 0.05

### ttl parameters
setting: offline_ttl
threshold: {OFFICIAL_CE_THRESHOLD}
lamb: 0.1

### dataset
dataset: {SAMPLE_KEY}
eval_dataset: {SAMPLE_KEY}
template: qwen
cutoff_len: 128
max_samples: {SAMPLE_SIZE}
overwrite_cache: true
preprocessing_num_workers: 1

### output
output_dir: saves/qwen1.5-0.5b/offline_ttl/agriculture_5k/{OFFICIAL_OUT_NAME}
logging_steps: 1
save_steps: 125
save_strategy: epoch
plot_loss: true
overwrite_output_dir: true

### train
seed: {TRAIN_SEED}
gradient_checkpointing: {str(GRADIENT_CHECKPOINTING).lower()}
per_device_train_batch_size: 1
gradient_accumulation_steps: 4
learning_rate: 5.0e-5
num_train_epochs: 1.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
bf16: true
ddp_timeout: 180000000

### eval
do_eval: null

### predict
temperature: 0.0
do_sample: false
max_new_tokens: {MAX_NEW_TOKENS}
per_device_eval_batch_size: 1
predict_with_generate: true

report_to: none
'''
    config_path = Path(f'examples/train_lora/offline_ttl_agri_{OFFICIAL_OUT_NAME}.yaml')
    config_path.parent.mkdir(parents=True, exist_ok=True)
    config_path.write_text(official_config)

    print('Running official CE:', OFFICIAL_OUT_NAME)
    !python src/train.py {config_path}
else:
    print('Skipping official baseline.')

Streaming output truncated to the last 5000 lines.
[WARNING|trainer.py:760] 2026-06-25 08:35:35,494 >> Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
[WARNING|trainer.py:760] 2026-06-25 08:35:35,494 >> Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
[WARNING|trainer.py:760] 2026-06-25 08:35:35,495 >> Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
 22% 108/500 [00:28<01:53,  3.47it/s][WARNING|trainer.py:760] 2026-06-25 08:35:35,495 >> Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
[WARNING|trainer.py:760] 2026-06-25 08:35:35,775 >> Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
[WARNING|trainer.py:760] 2026-06-25 08:35:35,775 >> Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
[WARNING|trainer.py:760] 2026-06-25 08:35:35,775 >> Trainer.tokenizer is now depreca

## 7.5 — Base Qwen no-TTL baseline

This cell generates predictions from the imported base Qwen1.5-0.5B model **without TTL training**, **without LoRA**, and **without adapter loading**.

It writes a `generated_predictions.jsonl` file in the same general format as the TLM prediction outputs, so the later comparison cell can include:

```text
base_qwen_no_ttl
official_current_ce
logit_grad_norm
logit_influence_cos
```

This is the regular base-model performance baseline.


In [69]:
# Cell 7.5 — Base Qwen no-TTL direct generation baseline
# FIXED: uses official CE prediction prompts + labels so references align exactly.

RUN_BASE_QWEN_NO_TTL = True

if RUN_BASE_QWEN_NO_TTL:
    import os, json, glob, torch, gc
    from pathlib import Path
    from tqdm.auto import tqdm
    from transformers import AutoTokenizer, AutoModelForCausalLM

    %cd /content/TLM

    torch.cuda.empty_cache()
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

    BASE_OUT = Path("/content/TLM/saves/qwen1.5-0.5b/offline_ttl/agriculture_5k")

    OFFICIAL_OUT_NAME = (
        f"OFFICIAL_current_ce_thresh{str(OFFICIAL_CE_THRESHOLD).replace('.', 'p')}"
        f"_max{MAX_NEW_TOKENS}_sample{SAMPLE_SIZE}_seed{DATA_SEED}"
    )

    BASE_QWEN_OUT_NAME = (
        f"BASE_QWEN_NO_TTL_max{MAX_NEW_TOKENS}"
        f"_sample{SAMPLE_SIZE}_seed{DATA_SEED}"
    )

    official_dir = BASE_OUT / OFFICIAL_OUT_NAME
    base_dir = BASE_OUT / BASE_QWEN_OUT_NAME
    base_dir.mkdir(parents=True, exist_ok=True)

    def find_predictions(output_dir):
        output_dir = Path(output_dir)
        patterns = [
            str(output_dir / "predict-*" / "generated_predictions.jsonl"),
            str(output_dir / "generated_predictions.jsonl"),
            str(output_dir / "**" / "generated_predictions.jsonl"),
        ]
        matches = []
        for p in patterns:
            matches.extend(glob.glob(p, recursive=True))
        matches = sorted(set(matches))
        if not matches:
            raise FileNotFoundError(
                f"No generated_predictions.jsonl found under {output_dir}. "
                "Run the official CE baseline cell before this base-Qwen cell."
            )
        return Path(matches[0])

    official_pred_path = find_predictions(official_dir)
    print("Using official CE predictions as prompt/reference source:")
    print(official_pred_path)

    official_rows = []
    with open(official_pred_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                official_rows.append(json.loads(line))

    if len(official_rows) != SAMPLE_SIZE and len(official_rows) % SAMPLE_SIZE == 0:
        print(f"WARNING: official file has {len(official_rows)} rows; using last {SAMPLE_SIZE}.")
        official_rows = official_rows[-SAMPLE_SIZE:]

    if len(official_rows) != SAMPLE_SIZE:
        raise ValueError(f"Expected {SAMPLE_SIZE} official rows, got {len(official_rows)}")

    prompts = [r.get("prompt", r.get("instruction", "")) for r in official_rows]
    refs = [r.get("label", r.get("reference", r.get("output", ""))) for r in official_rows]

    if not all(str(p).strip() for p in prompts):
        raise ValueError("Some official prompts are empty; cannot use them for base generation.")
    if not all(str(r).strip() for r in refs):
        print("WARNING: Some official labels are empty.")

    model_path = Path("/content/TLM/models/Qwen1.5-0.5B")
    if not (model_path / "config.json").exists():
        raise FileNotFoundError(f"Missing model config.json at {model_path}")

    print("Loading base Qwen model from:", model_path)
    tokenizer = AutoTokenizer.from_pretrained(
        model_path,
        local_files_only=True,
        trust_remote_code=True,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
    base_model = AutoModelForCausalLM.from_pretrained(
        model_path,
        local_files_only=True,
        trust_remote_code=True,
        torch_dtype=dtype,
        device_map="auto",
    )
    base_model.eval()

    @torch.no_grad()
    def generate_one(prompt_text):
        inputs = tokenizer(
            prompt_text,
            return_tensors="pt",
            truncation=True,
            max_length=128,
        )
        inputs = {k: v.to(base_model.device) for k, v in inputs.items()}

        generated = base_model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

        new_tokens = generated[0][inputs["input_ids"].shape[-1]:]
        return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    pred_path = base_dir / "generated_predictions.jsonl"

    out_rows = []
    for i, (prompt, ref) in enumerate(tqdm(list(zip(prompts, refs)), desc="Base Qwen no-TTL generation")):
        pred = generate_one(prompt)
        out_rows.append({
            "prompt": prompt,
            "label": ref,
            "predict": pred,
        })

        if (i + 1) % 50 == 0:
            pred_path.write_text(
                "\n".join(json.dumps(x, ensure_ascii=False) for x in out_rows) + "\n",
                encoding="utf-8",
            )

    pred_path.write_text(
        "\n".join(json.dumps(x, ensure_ascii=False) for x in out_rows) + "\n",
        encoding="utf-8",
    )

    print("✅ Base Qwen no-TTL predictions written to:")
    print(pred_path)
    print("rows:", len(out_rows))

    # Sanity check: base labels should exactly match official labels.
    check_rows = []
    with open(pred_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                check_rows.append(json.loads(line))

    base_refs = [r.get("label", "") for r in check_rows]
    assert base_refs == refs, "Base references still do not match official references."
    print("✅ Base references align exactly with official CE references.")

    del base_model
    gc.collect()
    torch.cuda.empty_cache()
else:
    print("Skipping base Qwen no-TTL baseline.")

/content/TLM
Using official CE predictions as prompt/reference source:
/content/TLM/saves/qwen1.5-0.5b/offline_ttl/agriculture_5k/OFFICIAL_current_ce_thresh3p537274_max32_sample500_seed2030/predict-temperature_0.0-max_new_tokens_32/generated_predictions.jsonl
Loading base Qwen model from: /content/TLM/models/Qwen1.5-0.5B


Base Qwen no-TTL generation:   0%|          | 0/500 [00:00<?, ?it/s]

✅ Base Qwen no-TTL predictions written to:
/content/TLM/saves/qwen1.5-0.5b/offline_ttl/agriculture_5k/BASE_QWEN_NO_TTL_max32_sample500_seed2030/generated_predictions.jsonl
rows: 500
✅ Base references align exactly with official CE references.


## 8 — Patch trainer.py: CE replacement via logit-gradient score

In [70]:
import os, re, py_compile
from pathlib import Path

%cd /content/TLM
trainer_path = Path('/content/TLM/src/llamafactory/train/ttl/trainer.py')
!git checkout -- src/llamafactory/train/ttl/trainer.py

text = trainer_path.read_text(encoding='utf-8')
if 'import os' not in text.split('\n')[:80]:
    if 'import math' in text:
        text = text.replace('import math', 'import math\nimport os', 1)
    else:
        text = 'import os\n' + text

custom_tail = r'''
    def _ttl_labels_for_logit_score(self, labels, input_ids=None, attention_mask=None):
        use_labels = labels
        if use_labels is None or (use_labels != IGNORE_INDEX).sum().item() == 0:
            if input_ids is None:
                raise RuntimeError("No valid labels and no input_ids available for logit-gradient scoring.")
            use_labels = input_ids.clone()
            if attention_mask is not None:
                use_labels = use_labels.masked_fill(attention_mask == 0, IGNORE_INDEX)
            if use_labels.size(-1) > 0:
                use_labels[:, 0] = IGNORE_INDEX
        return use_labels

    def _ttl_noise_token_id_mask(self, device, vocab_size):
        mode = os.environ.get("TLM_TOKEN_MASK_MODE", "none").strip().lower()
        if mode in ("", "none", "off", "0"):
            return torch.zeros(vocab_size, device=device, dtype=torch.bool)

        if mode not in ("punct_ws", "punctuation", "punctuation_whitespace"):
            raise ValueError(f"Unknown TLM_TOKEN_MASK_MODE={mode}")

        cache_name = "_ttl_noise_token_ids_cpu_punct_ws"
        if not hasattr(self, cache_name):
            import string
            import unicodedata

            tokenizer = getattr(self, "tokenizer", None)
            if tokenizer is None:
                tokenizer = getattr(self, "processing_class", None)

            bad_ids = []
            special_ids = set(getattr(tokenizer, "all_special_ids", []) or []) if tokenizer is not None else set()

            def is_punct_or_ws_text(text):
                if text is None:
                    return False
                text = str(text)
                if text == "" or text.strip() == "":
                    return True
                stripped = text.strip()
                if stripped == "":
                    return True
                for ch in stripped:
                    cat = unicodedata.category(ch)
                    if ch.isspace() or ch in string.punctuation or cat.startswith("P"):
                        continue
                    return False
                return True

            if tokenizer is None:
                bad_ids = []
            else:
                for tok_id in range(vocab_size):
                    if tok_id in special_ids:
                        bad_ids.append(tok_id)
                        continue
                    try:
                        piece = tokenizer.decode([tok_id], clean_up_tokenization_spaces=False, skip_special_tokens=False)
                    except Exception:
                        try:
                            piece = tokenizer.convert_ids_to_tokens(tok_id)
                        except Exception:
                            piece = ""
                    if is_punct_or_ws_text(piece):
                        bad_ids.append(tok_id)

            setattr(self, cache_name, torch.tensor(bad_ids, dtype=torch.long))
            print(f"[TLM token mask] mode={mode}, masked_token_ids={len(bad_ids)} / {vocab_size}")

        bad_ids_cpu = getattr(self, cache_name)
        out = torch.zeros(vocab_size, device=device, dtype=torch.bool)
        if bad_ids_cpu.numel() > 0:
            bad_ids = bad_ids_cpu.to(device)
            bad_ids = bad_ids[(bad_ids >= 0) & (bad_ids < vocab_size)]
            if bad_ids.numel() > 0:
                out[bad_ids] = True
        return out

    def _ttl_apply_token_type_mask(self, shift_labels, valid_mask, vocab_size):
        mode = os.environ.get("TLM_TOKEN_MASK_MODE", "none").strip().lower()
        if mode in ("", "none", "off", "0"):
            return valid_mask

        bad_vocab_mask = self._ttl_noise_token_id_mask(shift_labels.device, vocab_size)
        safe_labels = shift_labels.masked_fill(~valid_mask, 0).long()
        bad_positions = bad_vocab_mask[safe_labels]
        new_valid = valid_mask & (~bad_positions)

        if new_valid.sum().item() == 0 and valid_mask.sum().item() > 0:
            return valid_mask
        return new_valid

    @torch.no_grad()
    def _ttl_logit_grad_token_norm_score(self, logits, labels):
        shift_logits = logits[..., :-1, :].float()
        shift_labels = labels[..., 1:].contiguous()
        valid_mask = shift_labels != IGNORE_INDEX
        valid_mask = self._ttl_apply_token_type_mask(shift_labels, valid_mask, shift_logits.size(-1))
        if valid_mask.sum().item() == 0:
            return torch.zeros(shift_labels.size(0), device=logits.device)

        probs = torch.softmax(shift_logits, dim=-1)
        safe_labels = shift_labels.masked_fill(~valid_mask, 0)
        p_gold = probs.gather(-1, safe_labels.unsqueeze(-1)).squeeze(-1)
        sum_p2 = probs.pow(2).sum(dim=-1)
        token_norm = torch.sqrt(torch.clamp(sum_p2 + 1.0 - 2.0 * p_gold, min=0.0))
        token_norm = token_norm.masked_fill(~valid_mask, 0.0)

        top_frac = float(os.environ.get("TLM_LOGIT_GRAD_TOP_FRAC", "1.0"))
        top_frac = max(0.0, min(1.0, top_frac))
        scores = []
        for row_norms, row_mask in zip(token_norm, valid_mask):
            vals = row_norms[row_mask]
            if vals.numel() == 0:
                scores.append(torch.tensor(0.0, device=logits.device))
            elif top_frac >= 0.999:
                scores.append(vals.mean())
            else:
                k = int(torch.ceil(torch.tensor(float(vals.numel()) * top_frac)).item())
                k = max(1, min(k, vals.numel()))
                scores.append(torch.topk(vals, k=k, largest=True).values.mean())
        return torch.stack(scores)

    @torch.no_grad()
    def _ttl_logit_grad_mean_vector(self, logits, labels):
        shift_logits = logits[..., :-1, :].float()
        shift_labels = labels[..., 1:].contiguous()
        valid_mask = shift_labels != IGNORE_INDEX
        valid_mask = self._ttl_apply_token_type_mask(shift_labels, valid_mask, shift_logits.size(-1))
        probs = torch.softmax(shift_logits, dim=-1)
        vocab_size = shift_logits.size(-1)
        vectors = []

        for row_probs, row_labels, row_mask in zip(probs, shift_labels, valid_mask):
            valid_count = row_mask.sum().item()
            if valid_count == 0:
                vectors.append(torch.zeros(vocab_size, device=logits.device, dtype=torch.float32))
                continue
            valid_probs = row_probs[row_mask]
            valid_labels = row_labels[row_mask]
            p_mean = valid_probs.mean(dim=0)
            label_hist = torch.zeros(vocab_size, device=logits.device, dtype=torch.float32)
            label_hist.scatter_add_(
                0,
                valid_labels.long(),
                torch.ones_like(valid_labels, dtype=torch.float32) / float(valid_count),
            )
            vectors.append(p_mean - label_hist)
        return torch.stack(vectors)

    @torch.no_grad()
    def _ttl_logit_influence_cos_score(self, logits, labels):
        grad_vecs = self._ttl_logit_grad_mean_vector(logits, labels)
        eps = 1e-12
        if not hasattr(self, "_ttl_logit_ref_vec"):
            self._ttl_logit_ref_vec = None
        scores = []
        for vec in grad_vecs:
            vec = vec.detach()
            vec_norm = torch.norm(vec.float())
            if self._ttl_logit_ref_vec is None:
                score = torch.tensor(1.0, device=logits.device)
                self._ttl_logit_ref_vec = vec.detach().clone()
            else:
                ref = self._ttl_logit_ref_vec.to(vec.device)
                ref_norm = torch.norm(ref.float())
                if vec_norm.item() == 0.0 or ref_norm.item() == 0.0:
                    score = torch.tensor(0.0, device=logits.device)
                else:
                    score = torch.dot(vec.float(), ref.float()) / (vec_norm * ref_norm + eps)
                decay = float(os.environ.get("TLM_INFLUENCE_EMA_DECAY", "0.95"))
                decay = max(0.0, min(0.9999, decay))
                self._ttl_logit_ref_vec = (decay * ref.detach()) + ((1.0 - decay) * vec.detach())
            scores.append(score.detach())
        return torch.stack(scores)

    @torch.no_grad()
    def _ttl_custom_selection_score(self, logits, inputs):
        method = os.environ.get("TLM_TTL_SCORE_METHOD", "grad_norm").strip().lower()
        labels = self._ttl_labels_for_logit_score(
            inputs.get("labels"),
            inputs.get("input_ids"),
            inputs.get("attention_mask"),
        )
        if method == "grad_norm":
            score = self._ttl_logit_grad_token_norm_score(logits, labels)
        elif method == "influence_cos":
            score = self._ttl_logit_influence_cos_score(logits, labels)
        elif method == "raw_ce":
            score = self.cal_ce(logits, labels)
        else:
            raise ValueError(f"Unknown TLM_TTL_SCORE_METHOD={method}")

        fail_on_zero = os.environ.get("TLM_FAIL_ON_ZERO_SCORES", "1").strip() != "0"
        if fail_on_zero and method != "raw_ce" and torch.max(torch.abs(score)).item() == 0.0:
            raise RuntimeError(f"Custom TTL score collapsed to all zeros for method={method}.")
        return score

    def _ttl_effective_threshold(self, sentence_score):
        mode = os.environ.get("TLM_DYNAMIC_THRESHOLD_MODE", "none").strip().lower()
        static_threshold = torch.tensor(float(self.finetuning_args.threshold), device=sentence_score.device, dtype=sentence_score.dtype)

        if mode in ("", "none", "off", "0"):
            threshold = static_threshold
        elif mode in ("rolling_percentile", "history_percentile"):
            percentile = float(os.environ.get("TLM_DYNAMIC_THRESHOLD_PERCENTILE", "25.0"))
            percentile = max(0.0, min(100.0, percentile))
            min_history = int(float(os.environ.get("TLM_DYNAMIC_THRESHOLD_MIN_HISTORY", "16")))
            max_history = int(float(os.environ.get("TLM_DYNAMIC_THRESHOLD_MAX_HISTORY", "2048")))
            if not hasattr(self, "_ttl_score_history"):
                self._ttl_score_history = []

            history = self._ttl_score_history[-max_history:]
            if len(history) >= min_history:
                hist_tensor = torch.tensor(history, device=sentence_score.device, dtype=sentence_score.dtype)
                threshold = torch.quantile(hist_tensor, percentile / 100.0)
            else:
                threshold = static_threshold

            vals = sentence_score.detach().float().flatten()
            vals = vals[torch.isfinite(vals)]
            if vals.numel() > 0:
                new_vals = [float(x) for x in vals.cpu().tolist()]
                self._ttl_score_history = (self._ttl_score_history + new_vals)[-max_history:]
        else:
            raise ValueError(f"Unknown TLM_DYNAMIC_THRESHOLD_MODE={mode}")

        self._ttl_last_effective_threshold = float(threshold.detach().float().cpu().item())
        self._ttl_last_dynamic_threshold_mode = mode
        return threshold

    def _ttl_mask_and_coeff_from_score(self, sentence_score):
        sentence_score = sentence_score.clone().detach()
        threshold = self._ttl_effective_threshold(sentence_score)
        mask = sentence_score >= threshold
        score_exp_clip = float(os.environ.get("TLM_CUSTOM_SCORE_EXP_CLIP", "8.0"))
        clipped_delta = torch.clamp(sentence_score - threshold, max=score_exp_clip)
        coeff = self.finetuning_args.lamb * torch.exp(clipped_delta)
        return mask, coeff

    def _ttl_log_score_to_file(self, sentence_score, sentence_kl, mask, coeff, loss):
        method = os.environ.get("TLM_TTL_SCORE_METHOD", "raw_ce")
        token_mask_mode = os.environ.get("TLM_TOKEN_MASK_MODE", "none")
        dyn_mode = os.environ.get("TLM_DYNAMIC_THRESHOLD_MODE", "none")
        threshold_val = getattr(self, "_ttl_last_effective_threshold", float(self.finetuning_args.threshold))
        with open(f"{self.args.output_dir}/logfile.txt", "a", encoding="utf-8") as f:
            for score, kl, m, coef in zip(sentence_score.clone().detach(), sentence_kl.clone().detach(), mask, coeff):
                if m:
                    print(
                        f"This sample is selected. Threshold: {threshold_val}, "
                        f"Score method: {method}, Token mask: {token_mask_mode}, Dynamic threshold: {dyn_mode}, "
                        f"Selection score: {float(score)}, KL divergence: {float(kl)}, "
                        f"Weight coefficient: {float(coef)}, Final loss: {loss}",
                        file=f,
                    )
                else:
                    print(
                        f"This sample is discarded. Threshold: {threshold_val}, "
                        f"Score method: {method}, Token mask: {token_mask_mode}, Dynamic threshold: {dyn_mode}, "
                        f"Selection score: {float(score)}, KL divergence: {float(kl)}",
                        file=f,
                    )

    @override
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        assert self.tokenizer.padding_side == 'right', "Training should be done with right padding."
        score_method = os.environ.get("TLM_TTL_SCORE_METHOD", "raw_ce").strip().lower()

        if self.finetuning_args.setting == "offline_ttl":
            with torch.no_grad():
                model.eval()
                with self.accelerator.unwrap_model(model).disable_adapter():
                    pretrain_logits = model(**inputs).logits
                if score_method == "raw_ce":
                    sentence_score = self.cal_ce(pretrain_logits, inputs["labels"])
                else:
                    sentence_score = self._ttl_custom_selection_score(pretrain_logits, inputs)

            mask, coeff = self._ttl_mask_and_coeff_from_score(sentence_score)
            model.train()
            outputs = model(**inputs)
            sentence_kl = self.cal_kl(outputs.logits, inputs["labels"])
            sentence_kl = sentence_kl.mul(coeff).mul(mask)
            total_loss = sentence_kl.mean() if mask.sum() == 0 else sentence_kl.sum() / mask.sum()
            self._ttl_log_score_to_file(sentence_score, sentence_kl.clone().detach(), mask, coeff, total_loss.item())

        elif self.finetuning_args.setting == "online_ttl":
            model.train()
            outputs = model(**inputs)
            if score_method == "raw_ce":
                sentence_score = self.cal_ce(outputs.logits, inputs["labels"])
            else:
                sentence_score = self._ttl_custom_selection_score(outputs.logits, inputs)
            mask, coeff = self._ttl_mask_and_coeff_from_score(sentence_score)
            sentence_kl = self.cal_kl(outputs.logits, inputs["labels"])
            sentence_kl = sentence_kl.mul(coeff).mul(mask)
            total_loss = sentence_kl.mean() if mask.sum() == 0 else sentence_kl.sum() / mask.sum()
            self._ttl_log_score_to_file(sentence_score, sentence_kl.clone().detach(), mask, coeff, total_loss.item())
        else:
            outputs = model(**inputs)
            total_loss = outputs.loss

        if is_transformers_version_equal_to_4_46() and not getattr(self, "model_accepts_loss_kwargs", False):
            if return_outputs:
                return (total_loss / self.args.gradient_accumulation_steps, outputs)
            else:
                return total_loss / self.args.gradient_accumulation_steps
        return (total_loss, outputs) if return_outputs else total_loss
'''

pattern = re.compile(
    r"    @override\n"
    r"    def compute_loss\(self, model, inputs, return_outputs=False, \*\*kwargs\):\n"
    r".*\Z",
    re.DOTALL,
)
patched, n = pattern.subn(custom_tail.rstrip() + "\n", text)
if n != 1:
    raise RuntimeError(f'Expected to replace exactly one compute_loss tail, replaced {n}')

fixed = []
for line in patched.splitlines():
    if line.endswith(r"\n"):
        line = line[:-2]
    fixed.append(line)
patched = "\n".join(fixed) + "\n"

trainer_path.write_text(patched, encoding='utf-8')
py_compile.compile(str(trainer_path), doraise=True)
print('✅ Patched trainer.py with logit scores + token masking + dynamic thresholding and compiled cleanly.')
!grep -n "_ttl_custom_selection_score\\|_ttl_apply_token_type_mask\\|_ttl_effective_threshold" /content/TLM/src/llamafactory/train/ttl/trainer.py | head -30


/content/TLM
✅ Patched trainer.py with logit scores + token masking + dynamic thresholding and compiled cleanly.
291:    def _ttl_apply_token_type_mask(self, shift_labels, valid_mask, vocab_size):
310:        valid_mask = self._ttl_apply_token_type_mask(shift_labels, valid_mask, shift_logits.size(-1))
341:        valid_mask = self._ttl_apply_token_type_mask(shift_labels, valid_mask, shift_logits.size(-1))
390:    def _ttl_custom_selection_score(self, logits, inputs):
411:    def _ttl_effective_threshold(self, sentence_score):
446:        threshold = self._ttl_effective_threshold(sentence_score)
489:                    sentence_score = self._ttl_custom_selection_score(pretrain_logits, inputs)
505:                sentence_score = self._ttl_custom_selection_score(outputs.logits, inputs)


## 9 — Verify custom trainer patch

In [71]:
from pathlib import Path
import py_compile

%cd /content/TLM
trainer_path = Path('/content/TLM/src/llamafactory/train/ttl/trainer.py')
text = trainer_path.read_text(encoding='utf-8')
markers = ['_ttl_custom_selection_score', '_ttl_logit_grad_token_norm_score', '_ttl_logit_influence_cos_score', 'TLM_FAIL_ON_ZERO_SCORES']
for m in markers:
    print(m, '=>', m in text)
if not all(m in text for m in markers):
    raise RuntimeError('Patch missing expected markers.')
py_compile.compile(str(trainer_path), doraise=True)
print('✅ trainer.py compiles cleanly')

/content/TLM
_ttl_custom_selection_score => True
_ttl_logit_grad_token_norm_score => True
_ttl_logit_influence_cos_score => True
TLM_FAIL_ON_ZERO_SCORES => True
✅ trainer.py compiles cleanly


## 10 — Run TTL with logit-gradient norm selection

In [72]:
RUN_GRAD_NORM = True

if RUN_GRAD_NORM:
    import os, torch
    from pathlib import Path

    %cd /content/TLM
    SAMPLE_KEY = f'agriculture_{SAMPLE_SIZE}_seed{DATA_SEED}'

    def fmt(x):
        x = float(x)
        return str(int(x)) if x.is_integer() else str(x).replace('.', 'p')

    GRAD_NORM_OUT_NAME = (
        f"LOGITGRADNORM_thresh{fmt(GRAD_NORM_THRESHOLD)}"
        f"_top{int(LOGIT_GRAD_TOP_FRAC * 100)}"
        f"_clip{fmt(CUSTOM_SCORE_EXP_CLIP)}"
        f"_max{MAX_NEW_TOKENS}_sample{SAMPLE_SIZE}_seed{DATA_SEED}"
    )

    patch_check = Path('/content/TLM/src/llamafactory/train/ttl/trainer.py').read_text(encoding='utf-8')
    if '_ttl_custom_selection_score' not in patch_check:
        raise RuntimeError('Custom trainer patch is NOT active. Run Cell 8 first.')

    torch.cuda.empty_cache()
    os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
    os.environ['TLM_TTL_SCORE_METHOD'] = 'grad_norm'
    os.environ['TLM_CUSTOM_SCORE_EXP_CLIP'] = str(CUSTOM_SCORE_EXP_CLIP)
    os.environ['TLM_LOGIT_GRAD_TOP_FRAC'] = str(LOGIT_GRAD_TOP_FRAC)
    os.environ['TLM_FAIL_ON_ZERO_SCORES'] = '1'
    os.environ['TLM_TOKEN_MASK_MODE'] = 'none'
    os.environ['TLM_DYNAMIC_THRESHOLD_MODE'] = 'none'

    grad_config = f'''
### model
model_name_or_path: models/Qwen1.5-0.5B

### method
stage: ttl
do_train: true
do_predict: true
finetuning_type: lora
lora_target: q_proj,v_proj
trust_remote_code: true
lora_rank: 4
lora_dropout: 0.05

### ttl parameters
setting: offline_ttl
threshold: {GRAD_NORM_THRESHOLD}
lamb: 0.1

### dataset
dataset: {SAMPLE_KEY}
eval_dataset: {SAMPLE_KEY}
template: qwen
cutoff_len: 128
max_samples: {SAMPLE_SIZE}
overwrite_cache: true
preprocessing_num_workers: 1

### output
output_dir: saves/qwen1.5-0.5b/offline_ttl/agriculture_5k/{GRAD_NORM_OUT_NAME}
logging_steps: 1
save_steps: 125
save_strategy: epoch
plot_loss: true
overwrite_output_dir: true

### train
seed: {TRAIN_SEED}
gradient_checkpointing: {str(GRADIENT_CHECKPOINTING).lower()}
per_device_train_batch_size: 1
gradient_accumulation_steps: 4
learning_rate: 5.0e-5
num_train_epochs: 1.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
bf16: true
ddp_timeout: 180000000

### eval
do_eval: null

### predict
temperature: 0.0
do_sample: false
max_new_tokens: {MAX_NEW_TOKENS}
per_device_eval_batch_size: 1
predict_with_generate: true

report_to: none
'''
    config_path = Path(f'examples/train_lora/offline_ttl_agri_{GRAD_NORM_OUT_NAME}.yaml')
    config_path.parent.mkdir(parents=True, exist_ok=True)
    config_path.write_text(grad_config)

    print('Running logit-gradient norm TTL:', GRAD_NORM_OUT_NAME)
    !python src/train.py {config_path}
else:
    print('Skipping grad-norm run.')

Streaming output truncated to the last 5000 lines.
[WARNING|trainer.py:760] 2026-06-25 08:40:39,157 >> Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
[WARNING|trainer.py:760] 2026-06-25 08:40:39,157 >> Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
[WARNING|trainer.py:760] 2026-06-25 08:40:39,157 >> Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
 22% 108/500 [00:29<01:53,  3.46it/s][WARNING|trainer.py:760] 2026-06-25 08:40:39,158 >> Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
[WARNING|trainer.py:760] 2026-06-25 08:40:39,439 >> Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
[WARNING|trainer.py:760] 2026-06-25 08:40:39,439 >> Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
[WARNING|trainer.py:760] 2026-06-25 08:40:39,439 >> Trainer.tokenizer is now depreca

## 11 — Run TTL with logit-gradient influence-cosine selection

In [73]:
RUN_INFLUENCE_COS = True

if RUN_INFLUENCE_COS:
    import os, torch
    from pathlib import Path

    %cd /content/TLM
    SAMPLE_KEY = f'agriculture_{SAMPLE_SIZE}_seed{DATA_SEED}'

    def fmt(x):
        x = float(x)
        return str(int(x)) if x.is_integer() else str(x).replace('.', 'p')

    INFLUENCE_OUT_NAME = (
        f"LOGITINFLUENCECOS_thresh{fmt(INFLUENCE_COS_THRESHOLD)}"
        f"_ema{fmt(INFLUENCE_EMA_DECAY)}"
        f"_clip{fmt(CUSTOM_SCORE_EXP_CLIP)}"
        f"_max{MAX_NEW_TOKENS}_sample{SAMPLE_SIZE}_seed{DATA_SEED}"
    )

    patch_check = Path('/content/TLM/src/llamafactory/train/ttl/trainer.py').read_text(encoding='utf-8')
    if '_ttl_custom_selection_score' not in patch_check:
        raise RuntimeError('Custom trainer patch is NOT active. Run Cell 8 first.')

    torch.cuda.empty_cache()
    os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
    os.environ['TLM_TTL_SCORE_METHOD'] = 'influence_cos'
    os.environ['TLM_CUSTOM_SCORE_EXP_CLIP'] = str(CUSTOM_SCORE_EXP_CLIP)
    os.environ['TLM_INFLUENCE_EMA_DECAY'] = str(INFLUENCE_EMA_DECAY)
    os.environ['TLM_FAIL_ON_ZERO_SCORES'] = '1'
    os.environ['TLM_TOKEN_MASK_MODE'] = 'none'
    os.environ['TLM_DYNAMIC_THRESHOLD_MODE'] = 'none'

    infl_config = f'''
### model
model_name_or_path: models/Qwen1.5-0.5B

### method
stage: ttl
do_train: true
do_predict: true
finetuning_type: lora
lora_target: q_proj,v_proj
trust_remote_code: true
lora_rank: 4
lora_dropout: 0.05

### ttl parameters
setting: offline_ttl
threshold: {INFLUENCE_COS_THRESHOLD}
lamb: 0.1

### dataset
dataset: {SAMPLE_KEY}
eval_dataset: {SAMPLE_KEY}
template: qwen
cutoff_len: 128
max_samples: {SAMPLE_SIZE}
overwrite_cache: true
preprocessing_num_workers: 1

### output
output_dir: saves/qwen1.5-0.5b/offline_ttl/agriculture_5k/{INFLUENCE_OUT_NAME}
logging_steps: 1
save_steps: 125
save_strategy: epoch
plot_loss: true
overwrite_output_dir: true

### train
seed: {TRAIN_SEED}
gradient_checkpointing: {str(GRADIENT_CHECKPOINTING).lower()}
per_device_train_batch_size: 1
gradient_accumulation_steps: 4
learning_rate: 5.0e-5
num_train_epochs: 1.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
bf16: true
ddp_timeout: 180000000

### eval
do_eval: null

### predict
temperature: 0.0
do_sample: false
max_new_tokens: {MAX_NEW_TOKENS}
per_device_eval_batch_size: 1
predict_with_generate: true

report_to: none
'''
    config_path = Path(f'examples/train_lora/offline_ttl_agri_{INFLUENCE_OUT_NAME}.yaml')
    config_path.parent.mkdir(parents=True, exist_ok=True)
    config_path.write_text(infl_config)

    print('Running logit-influence TTL:', INFLUENCE_OUT_NAME)
    !python src/train.py {config_path}
else:
    print('Skipping influence run.')

Streaming output truncated to the last 5000 lines.
[WARNING|trainer.py:760] 2026-06-25 08:43:44,957 >> Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
[WARNING|trainer.py:760] 2026-06-25 08:43:44,957 >> Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
[WARNING|trainer.py:760] 2026-06-25 08:43:44,958 >> Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
 22% 108/500 [00:29<01:53,  3.46it/s][WARNING|trainer.py:760] 2026-06-25 08:43:44,958 >> Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
[WARNING|trainer.py:760] 2026-06-25 08:43:45,239 >> Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
[WARNING|trainer.py:760] 2026-06-25 08:43:45,239 >> Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
[WARNING|trainer.py:760] 2026-06-25 08:43:45,239 >> Trainer.tokenizer is now depreca

## 12 — Compare predictions across CE, logit-gradnorm, and logit-influence

In [14]:
import os, glob, json, hashlib, nltk, torch
import numpy as np
import pandas as pd
from pathlib import Path
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from bert_score import score as bert_score
from rouge_chinese import Rouge
import jieba

%cd /content/TLM
nltk.download('punkt')
nltk.download('punkt_tab')
torch.cuda.empty_cache()

BASE_OUT = Path('/content/TLM/saves/qwen1.5-0.5b/offline_ttl/agriculture_5k')

def fmt(x):
    x = float(x)
    return str(int(x)) if x.is_integer() else str(x).replace('.', 'p')

BASE_QWEN_OUT_NAME = (
    f"BASE_QWEN_NO_TTL_max{MAX_NEW_TOKENS}"
    f"_sample{SAMPLE_SIZE}_seed{DATA_SEED}"
)

OFFICIAL_OUT_NAME = (
    f"OFFICIAL_current_ce_thresh{str(OFFICIAL_CE_THRESHOLD).replace('.', 'p')}"
    f"_max{MAX_NEW_TOKENS}_sample{SAMPLE_SIZE}_seed{DATA_SEED}"
)
GRAD_NORM_OUT_NAME = (
    f"LOGITGRADNORM_thresh{fmt(GRAD_NORM_THRESHOLD)}"
    f"_top{int(LOGIT_GRAD_TOP_FRAC * 100)}"
    f"_clip{fmt(CUSTOM_SCORE_EXP_CLIP)}"
    f"_max{MAX_NEW_TOKENS}_sample{SAMPLE_SIZE}_seed{DATA_SEED}"
)
INFLUENCE_OUT_NAME = (
    f"LOGITINFLUENCECOS_thresh{fmt(INFLUENCE_COS_THRESHOLD)}"
    f"_ema{fmt(INFLUENCE_EMA_DECAY)}"
    f"_clip{fmt(CUSTOM_SCORE_EXP_CLIP)}"
    f"_max{MAX_NEW_TOKENS}_sample{SAMPLE_SIZE}_seed{DATA_SEED}"
)
RUNS = {
    'base_qwen_no_ttl': BASE_OUT / BASE_QWEN_OUT_NAME,
    'official_current_ce': BASE_OUT / OFFICIAL_OUT_NAME,
    'logit_grad_norm': BASE_OUT / GRAD_NORM_OUT_NAME,
    'logit_influence_cos': BASE_OUT / INFLUENCE_OUT_NAME,
}
EXPECTED_ROWS = SAMPLE_SIZE
TRIM_DUPLICATE_PREDICTIONS = True

for name, out_dir in RUNS.items():
    print(name, '=>', out_dir, 'exists:', out_dir.exists())

def find_predictions(output_dir):
    output_dir = Path(output_dir)
    patterns = [
        str(output_dir / 'predict-*' / 'generated_predictions.jsonl'),
        str(output_dir / 'generated_predictions.jsonl'),
        str(output_dir / '**' / 'generated_predictions.jsonl'),
    ]
    matches = []
    for p in patterns:
        matches.extend(glob.glob(p, recursive=True))
    matches = sorted(set(matches))
    if not matches:
        raise FileNotFoundError(f'No generated_predictions.jsonl found under {output_dir}')
    return matches[0]

def file_hash(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def maybe_trim_rows(rows, name, expected=EXPECTED_ROWS):
    if expected is None or len(rows) == expected:
        return rows
    if TRIM_DUPLICATE_PREDICTIONS and len(rows) % expected == 0:
        print('WARNING:', name, 'has', len(rows), 'rows; using last', expected)
        return rows[-expected:]
    raise ValueError(f'{name} has {len(rows)} rows, expected {expected}')

def load_preds(path, name):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    rows = maybe_trim_rows(rows, name)
    prompts = [r.get('prompt', r.get('instruction', '')) for r in rows]
    refs = [r.get('label', r.get('reference', r.get('output', ''))) for r in rows]
    preds = [r.get('predict', r.get('prediction', r.get('response', ''))) for r in rows]
    return rows, prompts, refs, preds

loaded = {}
for name, out_dir in RUNS.items():
    pred_path = find_predictions(out_dir)
    rows, prompts, refs, preds = load_preds(pred_path, name)
    loaded[name] = {'path': pred_path, 'hash': file_hash(pred_path), 'rows': rows, 'prompts': prompts, 'refs': refs, 'preds': preds}
    print('\n', name)
    print(' predictions:', pred_path)
    print(' hash:', loaded[name]['hash'])
    print(' rows:', len(rows))

references = loaded['official_current_ce']['refs']
for name, payload in loaded.items():
    if payload['refs'] != references:
        bad = [
            i for i, (a, b) in enumerate(zip(payload['refs'], references))
            if a != b
        ]
        print(f"Reference mismatch for {name}: {len(bad)} mismatched rows")
        if bad:
            i = bad[0]
            print("First mismatch idx:", i)
            print("This run ref:", repr(payload['refs'][i])[:500])
            print("Official ref:", repr(references[i])[:500])
        raise ValueError(f"Reference mismatch for {name}")

def rouge_chinese_l(candidate, reference):
    candidate = str(candidate).replace('<n>', ' ').strip()
    reference = str(reference).strip()
    if not candidate or not reference:
        return 0.0
    rouge = Rouge()
    hyp = ' '.join(jieba.cut(candidate))
    ref = ' '.join(jieba.cut(reference))
    try:
        return float(rouge.get_scores(hyp, ref)[0]['rouge-l']['f'])
    except Exception:
        return 0.0

def compute_metrics(candidates, references, name):
    print(f'\nComputing metrics for {name}')
    _, _, F1 = bert_score(candidates, references, lang='en', verbose=False)
    bert = F1.mean().item()
    smooth = SmoothingFunction().method4
    bleu_sum = 0.0
    for cand, ref in zip(candidates, references):
        bleu_sum += sentence_bleu([nltk.word_tokenize(str(ref))], nltk.word_tokenize(str(cand)), smoothing_function=smooth)
    bleu = bleu_sum / len(candidates)
    rouge_l = sum(rouge_chinese_l(c, r) for c, r in zip(candidates, references)) / len(candidates)
    return {'name': name, 'BERTScore': bert, 'BLEU': bleu, 'ROUGE-L_rouge_chinese': rouge_l}

metrics = {name: compute_metrics(payload['preds'], references, name) for name, payload in loaded.items()}
metrics_df = pd.DataFrame(metrics.values())
display(metrics_df)

base = metrics['official_current_ce']
delta_df = pd.DataFrame([
    {
        'variant': name,
        'delta_BERTScore_vs_CE': m['BERTScore'] - base['BERTScore'],
        'delta_BLEU_vs_CE': m['BLEU'] - base['BLEU'],
        'delta_ROUGE_L_rouge_chinese_vs_CE': m['ROUGE-L_rouge_chinese'] - base['ROUGE-L_rouge_chinese'],
    }
    for name, m in metrics.items() if name != 'official_current_ce'
])
display(delta_df)

/content/TLM
base_qwen_no_ttl => /content/TLM/saves/qwen1.5-0.5b/offline_ttl/agriculture_5k/BASE_QWEN_NO_TTL_max32_sample500_seed2026 exists: True
official_current_ce => /content/TLM/saves/qwen1.5-0.5b/offline_ttl/agriculture_5k/OFFICIAL_current_ce_thresh3p537274_max32_sample500_seed2026 exists: True
logit_grad_norm => /content/TLM/saves/qwen1.5-0.5b/offline_ttl/agriculture_5k/LOGITGRADNORM_thresh0p587426_top100_clip8_max32_sample500_seed2026 exists: True
logit_influence_cos => /content/TLM/saves/qwen1.5-0.5b/offline_ttl/agriculture_5k/LOGITINFLUENCECOS_thresh0p73421_ema0p95_clip8_max32_sample500_seed2026 exists: True

 base_qwen_no_ttl
 predictions: /content/TLM/saves/qwen1.5-0.5b/offline_ttl/agriculture_5k/BASE_QWEN_NO_TTL_max32_sample500_seed2026/generated_predictions.jsonl
 hash: 91d236d4ecef7ec17d0586ec0c00ab0016f6cfdbe438b69aa9e037fcd53745be
 rows: 500

 official_current_ce
 predictions: /content/TLM/saves/qwen1.5-0.5b/offline_ttl/agriculture_5k/OFFICIAL_current_ce_thresh3p537274

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Computing metrics for logit_influence_cos


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


,name,BERTScore,BLEU,ROUGE-L_rouge_chinese
0,base_qwen_no_ttl,0.848492,0.029898,0.146490
1,official_current_ce,0.847734,0.029791,0.144289
2,logit_grad_norm,0.847603,0.029459,0.143479
3,logit_influence_cos,0.848555,0.031615,0.147022


,variant,delta_BERTScore_vs_CE,delta_BLEU_vs_CE,delta_ROUGE_L_rouge_chinese_vs_CE
0,base_qwen_no_ttl,0.000758,0.000107,0.002201
1,logit_grad_norm,-0.000132,-0.000332,-0.000810
2,logit_influence_cos,0.000820,0.001824,0.002734


## 13 — Diagnostics: changed-only metrics and exact overlap

In [ ]:
def avg_words(xs):
    return sum(len(str(x).split()) for x in xs) / max(1, len(xs))

def empty_count(xs):
    return sum(1 for x in xs if len(str(x).strip()) == 0)

def unique_count(xs):
    return len(set(str(x) for x in xs))

diag_df = pd.DataFrame([
    {'run': name, 'avg_pred_words': avg_words(p['preds']), 'empty_outputs': empty_count(p['preds']), 'unique_outputs': unique_count(p['preds'])}
    for name, p in loaded.items()
])
display(diag_df)

run_names = list(loaded.keys())
overlap_rows = []
for a in run_names:
    for b in run_names:
        pa, pb = loaded[a]['preds'], loaded[b]['preds']
        same = sum(x == y for x, y in zip(pa, pb))
        overlap_rows.append({'a': a, 'b': b, 'same': same, 'different': len(pa) - same, 'same_pct': same / len(pa) * 100})
overlap_df = pd.DataFrame(overlap_rows)
display(overlap_df.pivot(index='a', columns='b', values='same_pct'))

official_preds = loaded['official_current_ce']['preds']
changed_metric_rows = []
for name in run_names:
    if name == 'official_current_ce':
        continue
    variant_preds = loaded[name]['preds']
    changed_idx = [i for i, (a, b) in enumerate(zip(official_preds, variant_preds)) if a != b]
    print('\n' + '=' * 90)
    print(name, 'changed examples:', len(changed_idx), '/', len(official_preds))
    if changed_idx:
        i = changed_idx[0]
        print('First changed idx:', i)
        print('Prompt:', loaded[name]['prompts'][i][:500])
        print('Reference:', references[i])
        print('Official:', official_preds[i])
        print(name + ':', variant_preds[i])

        official_changed = [official_preds[i] for i in changed_idx]
        variant_changed = [variant_preds[i] for i in changed_idx]
        refs_changed = [references[i] for i in changed_idx]
        official_m = compute_metrics(official_changed, refs_changed, f'official_changed_vs_{name}')
        variant_m = compute_metrics(variant_changed, refs_changed, f'{name}_changed_only')
        changed_metric_rows.append({
            'variant': name,
            'changed_n': len(changed_idx),
            'official_changed_BERTScore': official_m['BERTScore'],
            'variant_changed_BERTScore': variant_m['BERTScore'],
            'delta_changed_BERTScore': variant_m['BERTScore'] - official_m['BERTScore'],
            'official_changed_BLEU': official_m['BLEU'],
            'variant_changed_BLEU': variant_m['BLEU'],
            'delta_changed_BLEU': variant_m['BLEU'] - official_m['BLEU'],
            'official_changed_ROUGE_L': official_m['ROUGE-L_rouge_chinese'],
            'variant_changed_ROUGE_L': variant_m['ROUGE-L_rouge_chinese'],
            'delta_changed_ROUGE_L': variant_m['ROUGE-L_rouge_chinese'] - official_m['ROUGE-L_rouge_chinese'],
        })
changed_metrics_df = pd.DataFrame(changed_metric_rows)
display(changed_metrics_df)

## 14 — Training and score diagnostics

In [ ]:
import glob, json, re
from pathlib import Path
import numpy as np
import pandas as pd

BASE_OUT = Path('/content/TLM/saves/qwen1.5-0.5b/offline_ttl/agriculture_5k')

def find_file(output_dir, filename):
    return sorted(glob.glob(str(Path(output_dir) / '**' / filename), recursive=True))

def load_log_history(output_dir):
    state_files = find_file(output_dir, 'trainer_state.json')
    if state_files:
        return json.loads(Path(state_files[0]).read_text(encoding='utf-8')).get('log_history', [])
    jsonl_files = find_file(output_dir, 'trainer_log.jsonl')
    logs = []
    if jsonl_files:
        for line in Path(jsonl_files[0]).read_text(encoding='utf-8').splitlines():
            if line.strip():
                logs.append(json.loads(line))
    return logs

def summarize_training(output_dir):
    logs = load_log_history(output_dir)
    loss_entries = [x for x in logs if 'loss' in x]
    if not loss_entries:
        return {'has_logs': False, 'num_loss_logs': 0}
    losses = np.array([float(x['loss']) for x in loss_entries])
    grad_norms = np.array([float(x.get('grad_norm', 0.0) or 0.0) for x in loss_entries])
    nonzero_loss = losses > 0
    nonzero_grad = grad_norms > 0
    return {
        'has_logs': True,
        'num_loss_logs': len(losses),
        'zero_loss_count': int((losses == 0).sum()),
        'zero_loss_pct': float((losses == 0).mean() * 100),
        'nonzero_loss_count': int(nonzero_loss.sum()),
        'nonzero_loss_pct': float(nonzero_loss.mean() * 100),
        'mean_loss': float(losses.mean()),
        'mean_nonzero_loss': float(losses[nonzero_loss].mean()) if nonzero_loss.any() else 0.0,
        'max_loss': float(losses.max()),
        'zero_grad_count': int((grad_norms == 0).sum()),
        'zero_grad_pct': float((grad_norms == 0).mean() * 100),
        'nonzero_grad_count': int(nonzero_grad.sum()),
        'nonzero_grad_pct': float(nonzero_grad.mean() * 100),
        'mean_trainer_grad_norm': float(grad_norms.mean()),
        'max_trainer_grad_norm': float(grad_norms.max()),
    }

def parse_selection_log(output_dir):
    logfiles = find_file(output_dir, 'logfile.txt')
    if not logfiles:
        return {'has_logfile': False}
    text = Path(logfiles[0]).read_text(encoding='utf-8', errors='ignore')
    lines = [ln for ln in text.splitlines() if ln.strip()]
    selected = sum('is selected' in ln for ln in lines)
    discarded = sum('is discarded' in ln for ln in lines)
    score_vals, ce_vals = [], []
    for ln in lines:
        m = re.search(r'Selection score: ([^,]+)', ln)
        if m:
            try: score_vals.append(float(m.group(1).strip()))
            except Exception: pass
        m = re.search(r'Cross-entropy: ([^,]+)', ln)
        if m:
            try: ce_vals.append(float(m.group(1).strip()))
            except Exception: pass
    vals = score_vals if score_vals else ce_vals
    out = {'has_logfile': True, 'logfile': logfiles[0], 'lines': len(lines), 'selected': selected, 'discarded': discarded, 'selected_pct': selected / max(1, selected + discarded) * 100, 'score_count': len(vals)}
    if vals:
        arr = np.array(vals, dtype=float)
        out.update({
            'score_min': float(np.min(arr)),
            'score_p10': float(np.percentile(arr, 10)),
            'score_p25': float(np.percentile(arr, 25)),
            'score_mean': float(np.mean(arr)),
            'score_median': float(np.median(arr)),
            'score_p75': float(np.percentile(arr, 75)),
            'score_p90': float(np.percentile(arr, 90)),
            'score_max': float(np.max(arr)),
        })
    return out

summary_rows = []
for name, out_dir in RUNS.items():
    summary_rows.append({'run': name, 'dir': str(out_dir), **summarize_training(out_dir), **parse_selection_log(out_dir)})
run_summary_df = pd.DataFrame(summary_rows)
display(run_summary_df)
print('Use score percentiles to tune GRAD_NORM_THRESHOLD and INFLUENCE_COS_THRESHOLD.')

## 15 — Threshold tuning helper

In [ ]:
if 'run_summary_df' not in globals():
    raise RuntimeError('Run Cell 14 first.')
cols = ['run', 'selected_pct', 'score_min', 'score_p10', 'score_p25', 'score_median', 'score_p75', 'score_p90', 'score_max']
display(run_summary_df[[c for c in cols if c in run_summary_df.columns]])
print('If grad_norm selects too many, set GRAD_NORM_THRESHOLD to p75 or p90 and rerun Cell 10 onward.')
print('If influence selects too many, raise INFLUENCE_COS_THRESHOLD above 0.0 and rerun Cell 11 onward.')

## 16 — Package experiment into downloadable zip

In [ ]:
from pathlib import Path
import os, json, glob, shutil, zipfile, datetime, platform

%cd /content/TLM
RUN_LABEL = f'CE_LOGITGRAD_INFL_RANDOM_SAMPLE{SAMPLE_SIZE}_SEED{DATA_SEED}'
SAMPLE_KEY = f'agriculture_{SAMPLE_SIZE}_seed{DATA_SEED}'

EXPERIMENT_DETAILS = {
    'notebook': 'TLM_CE_vs_LogitGradNorm_Influence_RandomSampleSeeds_REMAKE.ipynb',
    'purpose': 'compare current TTL CE against logit-gradient norm and logit-gradient influence sample-selection scores',
    'sample_key': SAMPLE_KEY,
    'sample_size': SAMPLE_SIZE,
    'data_seed': DATA_SEED,
    'train_seed': TRAIN_SEED,
    'model': 'Qwen1.5-0.5B',
    'dataset_source': 'agriculture_5k randomized sample',
    'finetuning_type': 'lora',
    'lora_rank': 4,
    'lora_dropout': 0.05,
    'lora_target': 'q_proj,v_proj',
    'learning_rate': '5.0e-5',
    'num_train_epochs': 1.0,
    'lamb': 0.1,
    'max_new_tokens': MAX_NEW_TOKENS,
    'official_ce_threshold': OFFICIAL_CE_THRESHOLD,
    'grad_norm_threshold': GRAD_NORM_THRESHOLD,
    'influence_cos_threshold': INFLUENCE_COS_THRESHOLD,
    'custom_score_exp_clip': CUSTOM_SCORE_EXP_CLIP,
    'influence_ema_decay': INFLUENCE_EMA_DECAY,
    'logit_grad_top_frac': LOGIT_GRAD_TOP_FRAC,
    'run_dirs': {k: str(v) for k, v in RUNS.items()} if 'RUNS' in globals() else {},
}

def safe_copy(src, dst):
    src, dst = Path(src), Path(dst)
    if not src.exists():
        return False
    dst.parent.mkdir(parents=True, exist_ok=True)
    if src.is_dir():
        if dst.exists(): shutil.rmtree(dst)
        shutil.copytree(src, dst)
    else:
        shutil.copy2(src, dst)
    return True

def copy_matches(pattern, dst_dir, base_prefix=None):
    for f in sorted(glob.glob(str(pattern), recursive=True)):
        f = Path(f)
        try:
            rel = f.relative_to(base_prefix) if base_prefix else Path(f.name)
        except Exception:
            rel = Path(f.name)
        out = Path(dst_dir) / rel
        out.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(f, out)

def find_generated_predictions(run_dir):
    return [Path(x) for x in sorted(glob.glob(str(Path(run_dir) / '**' / 'generated_predictions.jsonl'), recursive=True))]

def find_adapter_files(run_dir):
    hits = []
    for pat in ['**/adapter_config.json', '**/adapter_model.safetensors', '**/adapter_model.bin']:
        hits.extend(glob.glob(str(Path(run_dir) / pat), recursive=True))
    return sorted(set(hits))

def to_jsonable(x):
    try:
        import numpy as np
        if isinstance(x, np.generic): return x.item()
    except Exception:
        pass
    if isinstance(x, Path): return str(x)
    return x

timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
package_root = Path(f'/content/{RUN_LABEL}_PACKAGE_{timestamp}')
if package_root.exists(): shutil.rmtree(package_root)
for sub in ['metadata', 'configs', 'dataset_sample', 'predictions', 'logs', 'adapters', 'source_patch']:
    (package_root / sub).mkdir(parents=True, exist_ok=True)

(package_root / 'metadata' / 'experiment_details.json').write_text(json.dumps(EXPERIMENT_DETAILS, indent=2, default=to_jsonable), encoding='utf-8')
metrics_payload = {}
for name in ['metrics_df', 'delta_df', 'diag_df', 'overlap_df', 'changed_metrics_df', 'run_summary_df']:
    if name in globals():
        obj = globals()[name]
        try: metrics_payload[name] = obj.to_dict('records')
        except Exception: metrics_payload[name] = str(obj)
(package_root / 'metadata' / 'metrics_summary.json').write_text(json.dumps(metrics_payload, indent=2, default=to_jsonable), encoding='utf-8')

copy_matches(Path('/content/TLM/examples/train_lora') / f'*sample{SAMPLE_SIZE}_seed{DATA_SEED}*.yaml', package_root / 'configs', base_prefix=Path('/content/TLM/examples/train_lora'))
sample_file = Path('/content/TLM/data/AdaptEval') / f'agriculture_random_sample_{SAMPLE_SIZE}_seed{DATA_SEED}.json'
manifest_file = Path('/content/TLM/data/AdaptEval') / f'agriculture_random_sample_{SAMPLE_SIZE}_seed{DATA_SEED}_indices.json'
safe_copy(sample_file, package_root / 'dataset_sample' / sample_file.name)
safe_copy(manifest_file, package_root / 'dataset_sample' / manifest_file.name)

if 'RUNS' in globals():
    for run_name, run_dir in RUNS.items():
        for p in find_generated_predictions(run_dir):
            safe_copy(p, package_root / 'predictions' / run_name / p.relative_to(run_dir))
        for pat in ['**/trainer_state.json', '**/trainer_log.jsonl', '**/training_args.bin', '**/all_results.json', '**/logfile.txt']:
            for p in sorted(glob.glob(str(Path(run_dir) / pat), recursive=True)):
                p = Path(p)
                safe_copy(p, package_root / 'logs' / run_name / p.relative_to(run_dir))
        for p in find_adapter_files(run_dir):
            p = Path(p)
            safe_copy(p, package_root / 'adapters' / run_name / p.relative_to(run_dir))

safe_copy(Path('/content/TLM/src/llamafactory/train/ttl/trainer.py'), package_root / 'source_patch' / 'trainer.py')
(package_root / 'README.md').write_text(f'''# TLM TTL CE vs Logit-Gradient Norm vs Logit-Influence Package

Sample key: `{SAMPLE_KEY}`
Sample size: `{SAMPLE_SIZE}`
Data seed: `{DATA_SEED}`
Train seed: `{TRAIN_SEED}`

Compared methods:
- official_current_ce: repo current CE
- logit_grad_norm: mean token norm of dCE/dlogits
- logit_influence_cos: cosine of mean logit-gradient vector vs EMA reference

Actual TTL adaptation loss remains the repo KL loss; only sample-selection score changes.
''', encoding='utf-8')

zip_path = Path(f'/content/{RUN_LABEL}_PACKAGE_{timestamp}.zip')
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for file in package_root.rglob('*'):
        if file.is_file():
            z.write(file, arcname=file.relative_to(package_root.parent))
print('✅ Package folder:', package_root)
print('✅ Zip file:', zip_path)
!du -sh {package_root}
!ls -lh {zip_path}
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as e:
    print('Could not auto-download. Use this path:', zip_path)

## 17 — How to rerun/tune

To test a new random sample, change `DATA_SEED` and `TRAIN_SEED`, then rerun from Cell 5 onward.

To tune custom selection:
1. Run the custom variant once.
2. Check Cell 14 score percentiles and selected percentage.
3. Adjust `GRAD_NORM_THRESHOLD`, `INFLUENCE_COS_THRESHOLD`, or `LOGIT_GRAD_TOP_FRAC`.
4. Rerun the custom run cell and Cells 12–16.

## 18 — AAAI upgrade tests: masking, dynamic thresholding, and forgetting check

Run this section after the original baseline + CE + GradNorm + Influence workflow is working.

This section adds a few targeted trains, not a random method explosion:
- punctuation/whitespace masking,
- rolling dynamic p25 thresholding,
- masking + dynamic p25,
- a CE dynamic control,
- a catastrophic-forgetting/general-language diagnostic.


In [74]:
# Cell 18.1 — Run targeted AAAI upgrade variants

RUN_AAAI_UPGRADE_TRAINS = True

AAAI_UPGRADE_VARIANTS = [
    {
        "name": "official_ce_dynp25",
        "score_method": "raw_ce",
        "threshold": OFFICIAL_CE_THRESHOLD,
        "token_mask_mode": "none",
        "dynamic_mode": "rolling_percentile",
        "dynamic_percentile": 25.0,
    },
    {
        "name": "influence_mask_punct_static75",
        "score_method": "influence_cos",
        "threshold": INFLUENCE_COS_THRESHOLD,
        "token_mask_mode": "punct_ws",
        "dynamic_mode": "none",
        "dynamic_percentile": 25.0,
    },
    {
        "name": "influence_dynp25",
        "score_method": "influence_cos",
        "threshold": INFLUENCE_COS_THRESHOLD,
        "token_mask_mode": "none",
        "dynamic_mode": "rolling_percentile",
        "dynamic_percentile": 25.0,
    },
    {
        "name": "influence_mask_punct_dynp25",
        "score_method": "influence_cos",
        "threshold": INFLUENCE_COS_THRESHOLD,
        "token_mask_mode": "punct_ws",
        "dynamic_mode": "rolling_percentile",
        "dynamic_percentile": 25.0,
    },
    {
        "name": "gradnorm_mask_punct_dynp25",
        "score_method": "grad_norm",
        "threshold": GRAD_NORM_THRESHOLD,
        "token_mask_mode": "punct_ws",
        "dynamic_mode": "rolling_percentile",
        "dynamic_percentile": 25.0,
    },
]

if RUN_AAAI_UPGRADE_TRAINS:
    import os, torch, re
    from pathlib import Path

    %cd /content/TLM
    SAMPLE_KEY = f'agriculture_{SAMPLE_SIZE}_seed{DATA_SEED}'
    BASE_OUT = Path('/content/TLM/saves/qwen1.5-0.5b/offline_ttl/agriculture_5k')

    def fmt(x):
        x = float(x)
        return str(int(x)) if x.is_integer() else str(x).replace('.', 'p')

    def safe_tag(s):
        return re.sub(r'[^A-Za-z0-9_]+', '_', str(s)).strip('_')

    patch_check = Path('/content/TLM/src/llamafactory/train/ttl/trainer.py').read_text(encoding='utf-8')
    required = ['_ttl_custom_selection_score', '_ttl_apply_token_type_mask', '_ttl_effective_threshold']
    missing = [x for x in required if x not in patch_check]
    if missing:
        raise RuntimeError(f'Enhanced trainer patch is not active. Missing: {missing}. Run Cell 8 first.')

    UPGRADE_RUNS = {}

    for v in AAAI_UPGRADE_VARIANTS:
        torch.cuda.empty_cache()

        name = safe_tag(v["name"])
        score_method = v["score_method"]
        threshold = float(v["threshold"])
        token_mask_mode = v.get("token_mask_mode", "none")
        dynamic_mode = v.get("dynamic_mode", "none")
        dynamic_percentile = float(v.get("dynamic_percentile", 25.0))

        out_name = (
            f"UPG_{name}"
            f"_method{safe_tag(score_method)}"
            f"_th{fmt(threshold)}"
            f"_mask{safe_tag(token_mask_mode)}"
            f"_dyn{safe_tag(dynamic_mode)}"
            f"_p{fmt(dynamic_percentile)}"
            f"_clip{fmt(CUSTOM_SCORE_EXP_CLIP)}"
            f"_max{MAX_NEW_TOKENS}_sample{SAMPLE_SIZE}_seed{DATA_SEED}"
        )

        os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
        os.environ['TLM_TTL_SCORE_METHOD'] = score_method
        os.environ['TLM_CUSTOM_SCORE_EXP_CLIP'] = str(CUSTOM_SCORE_EXP_CLIP)
        os.environ['TLM_LOGIT_GRAD_TOP_FRAC'] = str(LOGIT_GRAD_TOP_FRAC)
        os.environ['TLM_INFLUENCE_EMA_DECAY'] = str(INFLUENCE_EMA_DECAY)
        os.environ['TLM_FAIL_ON_ZERO_SCORES'] = '1'
        os.environ['TLM_TOKEN_MASK_MODE'] = token_mask_mode
        os.environ['TLM_DYNAMIC_THRESHOLD_MODE'] = dynamic_mode
        os.environ['TLM_DYNAMIC_THRESHOLD_PERCENTILE'] = str(dynamic_percentile)
        os.environ['TLM_DYNAMIC_THRESHOLD_MIN_HISTORY'] = str(DYNAMIC_THRESHOLD_MIN_HISTORY)
        os.environ['TLM_DYNAMIC_THRESHOLD_MAX_HISTORY'] = str(DYNAMIC_THRESHOLD_MAX_HISTORY)

        config = f'''
### model
model_name_or_path: models/Qwen1.5-0.5B

### method
stage: ttl
do_train: true
do_predict: true
finetuning_type: lora
lora_target: q_proj,v_proj
trust_remote_code: true
lora_rank: 4
lora_dropout: 0.05

### ttl parameters
setting: offline_ttl
threshold: {threshold}
lamb: 0.1

### dataset
dataset: {SAMPLE_KEY}
eval_dataset: {SAMPLE_KEY}
template: qwen
cutoff_len: 128
max_samples: {SAMPLE_SIZE}
overwrite_cache: true
preprocessing_num_workers: 1

### output
output_dir: saves/qwen1.5-0.5b/offline_ttl/agriculture_5k/{out_name}
logging_steps: 1
save_steps: 125
save_strategy: epoch
plot_loss: true
overwrite_output_dir: true

### train
seed: {TRAIN_SEED}
gradient_checkpointing: {str(GRADIENT_CHECKPOINTING).lower()}
per_device_train_batch_size: 1
gradient_accumulation_steps: 4
learning_rate: 5.0e-5
num_train_epochs: 1.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
bf16: true
ddp_timeout: 180000000

### eval
do_eval: null

### predict
temperature: 0.0
do_sample: false
max_new_tokens: {MAX_NEW_TOKENS}
per_device_eval_batch_size: 1
predict_with_generate: true

report_to: none
'''
        config_path = Path(f'examples/train_lora/offline_ttl_agri_{out_name}.yaml')
        config_path.parent.mkdir(parents=True, exist_ok=True)
        config_path.write_text(config)

        UPGRADE_RUNS[name] = BASE_OUT / out_name

        print('\n' + '=' * 100)
        print('Running upgrade variant:', name)
        print('score_method:', score_method, '| threshold:', threshold, '| token_mask:', token_mask_mode, '| dynamic:', dynamic_mode, '| percentile:', dynamic_percentile)
        print('output:', UPGRADE_RUNS[name])
        print('=' * 100)
        !python src/train.py {config_path}

    print('\n✅ Finished upgrade train cells.')
    for k, v in UPGRADE_RUNS.items():
        print(k, '=>', v)
else:
    print('Skipping AAAI upgrade trains.')


Streaming output truncated to the last 5000 lines.
[WARNING|trainer.py:760] 2026-06-25 08:58:50,188 >> Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
[WARNING|trainer.py:760] 2026-06-25 08:58:50,197 >> Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
[WARNING|trainer.py:760] 2026-06-25 08:58:50,197 >> Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
[WARNING|trainer.py:760] 2026-06-25 08:58:50,197 >> Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
[WARNING|trainer.py:760] 2026-06-25 08:58:50,197 >> Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
[WARNING|trainer.py:760] 2026-06-25 08:58:50,197 >> Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
 22% 109/500 [00:28<01:52,  3.46it/s][WARNING|trainer.py:760] 2026-06-25 08:58:50,198 >> Trainer.tokenizer is now depreca

In [75]:
# Cell 18.2 — Compare upgrade variants against official CE + current best methods

import os, glob, json, hashlib, nltk, torch
import numpy as np
import pandas as pd
from pathlib import Path
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from bert_score import score as bert_score
from rouge_chinese import Rouge
import jieba

%cd /content/TLM
nltk.download('punkt')
nltk.download('punkt_tab')
torch.cuda.empty_cache()

BASE_OUT = Path('/content/TLM/saves/qwen1.5-0.5b/offline_ttl/agriculture_5k')

def fmt(x):
    x = float(x)
    return str(int(x)) if x.is_integer() else str(x).replace('.', 'p')

BASE_QWEN_OUT_NAME = (
    f"BASE_QWEN_NO_TTL_max{MAX_NEW_TOKENS}"
    f"_sample{SAMPLE_SIZE}_seed{DATA_SEED}"
)
OFFICIAL_OUT_NAME = (
    f"OFFICIAL_current_ce_thresh{str(OFFICIAL_CE_THRESHOLD).replace('.', 'p')}"
    f"_max{MAX_NEW_TOKENS}_sample{SAMPLE_SIZE}_seed{DATA_SEED}"
)
GRAD_NORM_OUT_NAME = (
    f"LOGITGRADNORM_thresh{fmt(GRAD_NORM_THRESHOLD)}"
    f"_top{int(LOGIT_GRAD_TOP_FRAC * 100)}"
    f"_clip{fmt(CUSTOM_SCORE_EXP_CLIP)}"
    f"_max{MAX_NEW_TOKENS}_sample{SAMPLE_SIZE}_seed{DATA_SEED}"
)
INFLUENCE_OUT_NAME = (
    f"LOGITINFLUENCECOS_thresh{fmt(INFLUENCE_COS_THRESHOLD)}"
    f"_ema{fmt(INFLUENCE_EMA_DECAY)}"
    f"_clip{fmt(CUSTOM_SCORE_EXP_CLIP)}"
    f"_max{MAX_NEW_TOKENS}_sample{SAMPLE_SIZE}_seed{DATA_SEED}"
)

ALL_RUNS_FOR_UPGRADE_COMPARE = {
    'base_qwen_no_ttl': BASE_OUT / BASE_QWEN_OUT_NAME,
    'official_current_ce': BASE_OUT / OFFICIAL_OUT_NAME,
    'logit_grad_norm_static': BASE_OUT / GRAD_NORM_OUT_NAME,
    'logit_influence_cos_static': BASE_OUT / INFLUENCE_OUT_NAME,
}
if 'UPGRADE_RUNS' in globals():
    ALL_RUNS_FOR_UPGRADE_COMPARE.update({f'upgrade_{k}': Path(v) for k, v in UPGRADE_RUNS.items()})
else:
    print('WARNING: UPGRADE_RUNS not found. This cell will compare only existing baseline/static runs.')

EXPECTED_ROWS = SAMPLE_SIZE
TRIM_DUPLICATE_PREDICTIONS = True

def find_predictions(output_dir):
    output_dir = Path(output_dir)
    matches = []
    for p in [
        str(output_dir / 'predict-*' / 'generated_predictions.jsonl'),
        str(output_dir / 'generated_predictions.jsonl'),
        str(output_dir / '**' / 'generated_predictions.jsonl'),
    ]:
        matches.extend(glob.glob(p, recursive=True))
    matches = sorted(set(matches))
    if not matches:
        raise FileNotFoundError(f'No generated_predictions.jsonl found under {output_dir}')
    return matches[0]

def maybe_trim_rows(rows, name, expected=EXPECTED_ROWS):
    if expected is None or len(rows) == expected:
        return rows
    if TRIM_DUPLICATE_PREDICTIONS and len(rows) % expected == 0:
        print('WARNING:', name, 'has', len(rows), 'rows; using last', expected)
        return rows[-expected:]
    raise ValueError(f'{name} has {len(rows)} rows, expected {expected}')

def load_preds(path, name):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    rows = maybe_trim_rows(rows, name)
    prompts = [r.get('prompt', r.get('instruction', '')) for r in rows]
    refs = [r.get('label', r.get('reference', r.get('output', ''))) for r in rows]
    preds = [r.get('predict', r.get('prediction', r.get('response', ''))) for r in rows]
    return rows, prompts, refs, preds

loaded_upgrade = {}
for name, out_dir in ALL_RUNS_FOR_UPGRADE_COMPARE.items():
    print(name, '=>', out_dir, 'exists:', Path(out_dir).exists())
    pred_path = find_predictions(out_dir)
    rows, prompts, refs, preds = load_preds(pred_path, name)
    loaded_upgrade[name] = {'path': pred_path, 'rows': rows, 'prompts': prompts, 'refs': refs, 'preds': preds}
    print('  predictions:', pred_path, 'rows:', len(rows))

references_upgrade = loaded_upgrade['official_current_ce']['refs']
for name, payload in loaded_upgrade.items():
    if payload['refs'] != references_upgrade:
        bad = [i for i, (a, b) in enumerate(zip(payload['refs'], references_upgrade)) if a != b]
        print(f"Reference mismatch for {name}: {len(bad)} mismatched rows")
        if bad:
            i = bad[0]
            print("First mismatch idx:", i)
            print("This run ref:", repr(payload['refs'][i])[:500])
            print("Official ref:", repr(references_upgrade[i])[:500])
        raise ValueError(f"Reference mismatch for {name}")

def rouge_chinese_l(candidate, reference):
    candidate = str(candidate).replace('<n>', ' ').strip()
    reference = str(reference).strip()
    if not candidate or not reference:
        return 0.0
    rouge = Rouge()
    hyp = ' '.join(jieba.cut(candidate))
    ref = ' '.join(jieba.cut(reference))
    try:
        return float(rouge.get_scores(hyp, ref)[0]['rouge-l']['f'])
    except Exception:
        return 0.0

def compute_metrics(candidates, references, name):
    print(f'\nComputing metrics for {name}')
    _, _, F1 = bert_score(candidates, references, lang='en', verbose=False)
    bert = F1.mean().item()
    smooth = SmoothingFunction().method4
    bleu_sum = 0.0
    for cand, ref in zip(candidates, references):
        bleu_sum += sentence_bleu([nltk.word_tokenize(str(ref))], nltk.word_tokenize(str(cand)), smoothing_function=smooth)
    bleu = bleu_sum / len(candidates)
    rouge_l = sum(rouge_chinese_l(c, r) for c, r in zip(candidates, references)) / len(candidates)
    return {'name': name, 'BERTScore': bert, 'BLEU': bleu, 'ROUGE-L_rouge_chinese': rouge_l}

upgrade_metrics = {name: compute_metrics(payload['preds'], references_upgrade, name) for name, payload in loaded_upgrade.items()}
upgrade_metrics_df = pd.DataFrame(upgrade_metrics.values()).sort_values(
    ['ROUGE-L_rouge_chinese', 'BERTScore', 'BLEU'], ascending=False
)
display(upgrade_metrics_df)

ce_base = upgrade_metrics['official_current_ce']
upgrade_delta_df = pd.DataFrame([
    {
        'variant': name,
        'delta_BERTScore_vs_CE': m['BERTScore'] - ce_base['BERTScore'],
        'delta_BLEU_vs_CE': m['BLEU'] - ce_base['BLEU'],
        'delta_ROUGE_L_rouge_chinese_vs_CE': m['ROUGE-L_rouge_chinese'] - ce_base['ROUGE-L_rouge_chinese'],
    }
    for name, m in upgrade_metrics.items() if name != 'official_current_ce'
]).sort_values(['delta_ROUGE_L_rouge_chinese_vs_CE', 'delta_BERTScore_vs_CE'], ascending=False)
display(upgrade_delta_df)

print('\nBest by metric:')
for metric in ['BERTScore', 'BLEU', 'ROUGE-L_rouge_chinese']:
    row = upgrade_metrics_df.sort_values(metric, ascending=False).iloc[0]
    print(metric, '=>', row['name'], row[metric])


/content/TLM
base_qwen_no_ttl => /content/TLM/saves/qwen1.5-0.5b/offline_ttl/agriculture_5k/BASE_QWEN_NO_TTL_max32_sample500_seed2030 exists: True
  predictions: /content/TLM/saves/qwen1.5-0.5b/offline_ttl/agriculture_5k/BASE_QWEN_NO_TTL_max32_sample500_seed2030/generated_predictions.jsonl rows: 500
official_current_ce => /content/TLM/saves/qwen1.5-0.5b/offline_ttl/agriculture_5k/OFFICIAL_current_ce_thresh3p537274_max32_sample500_seed2030 exists: True
  predictions: /content/TLM/saves/qwen1.5-0.5b/offline_ttl/agriculture_5k/OFFICIAL_current_ce_thresh3p537274_max32_sample500_seed2030/predict-temperature_0.0-max_new_tokens_32/generated_predictions.jsonl rows: 500
logit_grad_norm_static => /content/TLM/saves/qwen1.5-0.5b/offline_ttl/agriculture_5k/LOGITGRADNORM_thresh0p587426_top100_clip8_max32_sample500_seed2030 exists: True
  predictions: /content/TLM/saves/qwen1.5-0.5b/offline_ttl/agriculture_5k/LOGITGRADNORM_thresh0p587426_top100_clip8_max32_sample500_seed2030/predict-temperature_0.0-

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Computing metrics for official_current_ce


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Computing metrics for logit_grad_norm_static


...


Computing metrics for upgrade_gradnorm_mask_punct_dynp25


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


,name,BERTScore,BLEU,ROUGE-L_rouge_chinese
8,upgrade_gradnorm_mask_punct_dynp25,0.846995,0.027226,0.137085
3,logit_influence_cos_static,0.846484,0.026497,0.136561
6,upgrade_influence_dynp25,0.846414,0.026989,0.136429
2,logit_grad_norm_static,0.846723,0.026749,0.135569
1,official_current_ce,0.846840,0.026622,0.134678
4,upgrade_official_ce_dynp25,0.846493,0.026279,0.134168
5,upgrade_influence_mask_punct_static75,0.846482,0.026248,0.133661
0,base_qwen_no_ttl,0.846326,0.025999,0.133450
7,upgrade_influence_mask_punct_dynp25,0.846714,0.026255,0.133323


,variant,delta_BERTScore_vs_CE,delta_BLEU_vs_CE,delta_ROUGE_L_rouge_chinese_vs_CE
7,upgrade_gradnorm_mask_punct_dynp25,0.000155,0.000604,0.002407
2,logit_influence_cos_static,-0.000356,-0.000125,0.001883
5,upgrade_influence_dynp25,-0.000426,0.000367,0.001751
1,logit_grad_norm_static,-0.000117,0.000127,0.000891
3,upgrade_official_ce_dynp25,-0.000347,-0.000343,-0.000510
4,upgrade_influence_mask_punct_static75,-0.000358,-0.000374,-0.001016
0,base_qwen_no_ttl,-0.000514,-0.000623,-0.001227
6,upgrade_influence_mask_punct_dynp25,-0.000126,-0.000367,-0.001355



Best by metric:
BERTScore => upgrade_gradnorm_mask_punct_dynp25 0.8469953536987305
BLEU => upgrade_gradnorm_mask_punct_dynp25 0.02722597543041276
ROUGE-L_rouge_chinese => upgrade_gradnorm_mask_punct_dynp25 0.13708465212682938


In [76]:
# Cell 18.3 — Upgrade training / score diagnostics

import glob, json, re
from pathlib import Path
import numpy as np
import pandas as pd

def find_file(output_dir, filename):
    return sorted(glob.glob(str(Path(output_dir) / '**' / filename), recursive=True))

def load_log_history(output_dir):
    state_files = find_file(output_dir, 'trainer_state.json')
    if state_files:
        return json.loads(Path(state_files[0]).read_text(encoding='utf-8')).get('log_history', [])
    jsonl_files = find_file(output_dir, 'trainer_log.jsonl')
    logs = []
    if jsonl_files:
        for line in Path(jsonl_files[0]).read_text(encoding='utf-8').splitlines():
            if line.strip():
                logs.append(json.loads(line))
    return logs

def summarize_training(output_dir):
    logs = load_log_history(output_dir)
    loss_entries = [x for x in logs if 'loss' in x]
    if not loss_entries:
        return {'has_logs': False, 'num_loss_logs': 0}
    losses = np.array([float(x['loss']) for x in loss_entries])
    grad_norms = np.array([float(x.get('grad_norm', 0.0) or 0.0) for x in loss_entries])
    return {
        'has_logs': True,
        'num_loss_logs': len(losses),
        'zero_loss_count': int((losses == 0).sum()),
        'zero_loss_pct': float((losses == 0).mean() * 100),
        'mean_loss': float(losses.mean()),
        'max_loss': float(losses.max()),
        'mean_trainer_grad_norm': float(grad_norms.mean()),
        'max_trainer_grad_norm': float(grad_norms.max()),
    }

def parse_selection_log(output_dir):
    logfiles = find_file(output_dir, 'logfile.txt')
    if not logfiles:
        return {'has_logfile': False}
    text = Path(logfiles[0]).read_text(encoding='utf-8', errors='ignore')
    lines = [ln for ln in text.splitlines() if ln.strip()]
    selected = sum('is selected' in ln for ln in lines)
    discarded = sum('is discarded' in ln for ln in lines)
    score_vals, threshold_vals = [], []
    for ln in lines:
        m = re.search(r'Selection score: ([^,]+)', ln)
        if m:
            try: score_vals.append(float(m.group(1).strip()))
            except Exception: pass
        m = re.search(r'Threshold: ([^,]+)', ln)
        if m:
            try: threshold_vals.append(float(m.group(1).strip()))
            except Exception: pass
    out = {
        'has_logfile': True,
        'logfile': logfiles[0],
        'lines': len(lines),
        'selected': selected,
        'discarded': discarded,
        'selected_pct': selected / max(1, selected + discarded) * 100,
        'score_count': len(score_vals),
    }
    if score_vals:
        arr = np.array(score_vals, dtype=float)
        out.update({
            'score_min': float(np.min(arr)),
            'score_p25': float(np.percentile(arr, 25)),
            'score_median': float(np.median(arr)),
            'score_p75': float(np.percentile(arr, 75)),
            'score_max': float(np.max(arr)),
        })
    if threshold_vals:
        th = np.array(threshold_vals, dtype=float)
        out.update({
            'threshold_mean': float(np.mean(th)),
            'threshold_min': float(np.min(th)),
            'threshold_max': float(np.max(th)),
        })
    return out

if 'ALL_RUNS_FOR_UPGRADE_COMPARE' not in globals():
    raise RuntimeError('Run Cell 18.2 first.')

diag_rows = []
for name, out_dir in ALL_RUNS_FOR_UPGRADE_COMPARE.items():
    if name == 'base_qwen_no_ttl':
        continue
    diag_rows.append({'run': name, 'dir': str(out_dir), **summarize_training(out_dir), **parse_selection_log(out_dir)})

upgrade_run_summary_df = pd.DataFrame(diag_rows)
display(upgrade_run_summary_df)


,run,dir,has_logs,num_loss_logs,zero_loss_count,zero_loss_pct,mean_loss,max_loss,mean_trainer_grad_norm,max_trainer_grad_norm,...,selected_pct,score_count,threshold_mean,threshold_min,threshold_max,score_min,score_p25,score_median,score_p75,score_max
0,official_current_ce,/content/TLM/saves/qwen1.5-0.5b/offline_ttl/ag...,True,125,0,0.0,1.085914,2.9692,1.160087,2.534877,...,75.4,0,3.537274,3.537274,3.537274,NaN,NaN,NaN,NaN,NaN
1,logit_grad_norm_static,/content/TLM/saves/qwen1.5-0.5b/offline_ttl/ag...,True,125,0,0.0,0.760300,1.6956,0.802989,2.056959,...,74.0,500,0.587426,0.587426,0.587426,0.537762,0.586956,0.607556,0.630700,0.697149
2,logit_influence_cos_static,/content/TLM/saves/qwen1.5-0.5b/offline_ttl/ag...,True,125,2,1.6,0.709169,1.6863,0.843526,2.124127,...,72.6,500,0.734210,0.734210,0.734210,0.488414,0.729198,0.775415,0.814725,1.000000
3,upgrade_official_ce_dynp25,/content/TLM/saves/qwen1.5-0.5b/offline_ttl/ag...,True,125,0,0.0,1.079003,2.9692,1.141384,2.488488,...,75.6,500,3.535616,3.474747,3.696837,2.785616,3.540497,3.758755,3.918539,4.761971
4,upgrade_influence_mask_punct_static75,/content/TLM/saves/qwen1.5-0.5b/offline_ttl/ag...,True,125,12,9.6,0.452598,1.1863,0.574578,1.667324,...,46.8,500,0.734210,0.734210,0.734210,0.470642,0.681068,0.726258,0.774961,1.000000
5,upgrade_influence_dynp25,/content/TLM/saves/qwen1.5-0.5b/offline_ttl/ag...,True,125,2,1.6,0.801429,1.8641,0.926467,2.230875,...,80.2,500,0.715028,0.630433,0.734210,0.488414,0.729198,0.775415,0.814725,1.000000
6,upgrade_influence_mask_punct_dynp25,/content/TLM/saves/qwen1.5-0.5b/offline_ttl/ag...,True,125,3,2.4,0.800714,1.8429,0.933633,2.254425,...,79.6,500,0.668751,0.596029,0.734210,0.470642,0.681068,0.726258,0.774961,1.000000
7,upgrade_gradnorm_mask_punct_dynp25,/content/TLM/saves/qwen1.5-0.5b/offline_ttl/ag...,True,125,1,0.8,0.803046,1.9283,0.859401,1.919147,...,77.2,500,0.701161,0.587426,0.709481,0.617863,0.706893,0.728802,0.754237,0.819886


In [77]:
# Cell 18.4 — Catastrophic forgetting / general-language check
# This is intentionally small. It is a reviewer-defense diagnostic, not the headline metric.

RUN_FORGETTING_CHECK = True  # flip to True near final evaluation

if RUN_FORGETTING_CHECK:
    import gc, math, torch
    import pandas as pd
    from pathlib import Path
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from peft import PeftModel

    %cd /content/TLM

    GENERAL_TEXTS = [
        "The history of science includes many examples of careful observation leading to new theories.",
        "A city council meeting was held on Tuesday to discuss transportation, housing, and public safety.",
        "The novel follows a young traveler who learns that courage often comes from helping other people.",
        "Economic growth can depend on investment, productivity, education, and stable institutions.",
        "Rainfall in the region varies by season, with the heaviest storms usually occurring in late spring.",
    ]

    MODEL_PATH = "/content/TLM/models/Qwen1.5-0.5B"

    def find_adapter_root(run_dir):
        run_dir = Path(run_dir)
        hits = sorted(run_dir.glob("**/adapter_config.json"))
        return hits[0].parent if hits else None

    @torch.no_grad()
    def mean_causal_lm_loss(model, tokenizer, texts, max_length=256):
        losses = []
        model.eval()
        for text in texts:
            batch = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length).to(model.device)
            out = model(**batch, labels=batch["input_ids"])
            losses.append(float(out.loss.detach().cpu()))
        return sum(losses) / len(losses)

    if 'ALL_RUNS_FOR_UPGRADE_COMPARE' not in globals():
        raise RuntimeError('Run Cell 18.2 first so ALL_RUNS_FOR_UPGRADE_COMPARE exists.')

    forgetting_rows = []

    for name, run_dir in ALL_RUNS_FOR_UPGRADE_COMPARE.items():
        adapter_root = None if name == 'base_qwen_no_ttl' else find_adapter_root(run_dir)

        print("\nLoading for forgetting check:", name, "adapter:", adapter_root)
        tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True, local_files_only=True)
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_PATH,
            trust_remote_code=True,
            local_files_only=True,
            torch_dtype=torch.bfloat16,
            device_map="auto",
        )
        if adapter_root is not None:
            model = PeftModel.from_pretrained(model, str(adapter_root))

        loss = mean_causal_lm_loss(model, tokenizer, GENERAL_TEXTS)
        ppl = math.exp(min(loss, 20.0))
        forgetting_rows.append({"run": name, "general_lm_loss_5sample": loss, "general_ppl_5sample": ppl})

        del model
        gc.collect()
        torch.cuda.empty_cache()

    forgetting_df = pd.DataFrame(forgetting_rows).sort_values("general_lm_loss_5sample")
    display(forgetting_df)

    base_loss = float(forgetting_df[forgetting_df["run"] == "base_qwen_no_ttl"]["general_lm_loss_5sample"].iloc[0])
    forgetting_delta_df = forgetting_df.copy()
    forgetting_delta_df["delta_general_lm_loss_vs_base"] = forgetting_delta_df["general_lm_loss_5sample"] - base_loss
    display(forgetting_delta_df.sort_values("delta_general_lm_loss_vs_base"))
else:
    print("Skipping forgetting check. Set RUN_FORGETTING_CHECK=True when you want the reviewer-shield diagnostic.")


/content/TLM

Loading for forgetting check: base_qwen_no_ttl adapter: None

Loading for forgetting check: official_current_ce adapter: /content/TLM/saves/qwen1.5-0.5b/offline_ttl/agriculture_5k/OFFICIAL_current_ce_thresh3p537274_max32_sample500_seed2030

Loading for forgetting check: logit_grad_norm_static adapter: /content/TLM/saves/qwen1.5-0.5b/offline_ttl/agriculture_5k/LOGITGRADNORM_thresh0p587426_top100_clip8_max32_sample500_seed2030

Loading for forgetting check: logit_influence_cos_static adapter: /content/TLM/saves/qwen1.5-0.5b/offline_ttl/agriculture_5k/LOGITINFLUENCECOS_thresh0p73421_ema0p95_clip8_max32_sample500_seed2030

Loading for forgetting check: upgrade_official_ce_dynp25 adapter: /content/TLM/saves/qwen1.5-0.5b/offline_ttl/agriculture_5k/UPG_official_ce_dynp25_methodraw_ce_th3p537274_masknone_dynrolling_percentile_p25_clip8_max32_sample500_seed2030

Loading for forgetting check: upgrade_influence_mask_punct_static75 adapter: /content/TLM/saves/qwen1.5-0.5b/offline_ttl

,run,general_lm_loss_5sample,general_ppl_5sample
0,base_qwen_no_ttl,3.311187,27.417639
2,logit_grad_norm_static,3.358972,28.759603
1,official_current_ce,3.367663,29.010653
7,upgrade_influence_mask_punct_dynp25,3.368536,29.035979
8,upgrade_gradnorm_mask_punct_dynp25,3.369709,29.070056
5,upgrade_influence_mask_punct_static75,3.372603,29.154331
4,upgrade_official_ce_dynp25,3.373566,29.182413
3,logit_influence_cos_static,3.376560,29.269901
6,upgrade_influence_dynp25,3.385229,29.524749


,run,general_lm_loss_5sample,general_ppl_5sample,delta_general_lm_loss_vs_base
0,base_qwen_no_ttl,3.311187,27.417639,0.000000
2,logit_grad_norm_static,3.358972,28.759603,0.047785
1,official_current_ce,3.367663,29.010653,0.056477
7,upgrade_influence_mask_punct_dynp25,3.368536,29.035979,0.057349
8,upgrade_gradnorm_mask_punct_dynp25,3.369709,29.070056,0.058522
5,upgrade_influence_mask_punct_static75,3.372603,29.154331,0.061417
4,upgrade_official_ce_dynp25,3.373566,29.182413,0.062380
3,logit_influence_cos_static,3.376560,29.269901,0.065373
6,upgrade_influence_dynp25,3.385229,29.524749,0.074042


In [21]:
# Cell 18.5 — Save upgrade summary artifacts

from pathlib import Path
import json, datetime
import pandas as pd

%cd /content/TLM

timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
summary_dir = Path(f'/content/TLM_AAAI_UPGRADE_SUMMARY_seed{DATA_SEED}_{timestamp}')
summary_dir.mkdir(parents=True, exist_ok=True)

payload = {
    "data_seed": DATA_SEED,
    "train_seed": TRAIN_SEED,
    "sample_size": SAMPLE_SIZE,
    "max_new_tokens": MAX_NEW_TOKENS,
    "official_ce_threshold": OFFICIAL_CE_THRESHOLD,
    "grad_norm_threshold": GRAD_NORM_THRESHOLD,
    "influence_cos_threshold": INFLUENCE_COS_THRESHOLD,
    "upgrade_variants": AAAI_UPGRADE_VARIANTS if 'AAAI_UPGRADE_VARIANTS' in globals() else [],
    "run_dirs": {k: str(v) for k, v in ALL_RUNS_FOR_UPGRADE_COMPARE.items()} if 'ALL_RUNS_FOR_UPGRADE_COMPARE' in globals() else {},
}
(summary_dir / "experiment_config.json").write_text(json.dumps(payload, indent=2), encoding="utf-8")

for name in [
    "upgrade_metrics_df",
    "upgrade_delta_df",
    "upgrade_run_summary_df",
    "forgetting_df",
    "forgetting_delta_df",
]:
    if name in globals():
        obj = globals()[name]
        try:
            obj.to_csv(summary_dir / f"{name}.csv", index=False)
        except Exception as e:
            print("Could not save", name, e)

print("Saved upgrade summary to:", summary_dir)
!ls -la {summary_dir}


/content/TLM
Saved upgrade summary to: /content/TLM_AAAI_UPGRADE_SUMMARY_seed2026_20260625_061650
total 36
drwxr-xr-x 2 root root 4096 Jun 25 06:16 .
drwxr-xr-x 1 root root 4096 Jun 25 06:16 ..
-rw-r--r-- 1 root root 3113 Jun 25 06:16 experiment_config.json
-rw-r--r-- 1 root root  829 Jun 25 06:16 forgetting_delta_df.csv
-rw-r--r-- 1 root root  631 Jun 25 06:16 forgetting_df.csv
-rw-r--r-- 1 root root  845 Jun 25 06:16 upgrade_delta_df.csv
-rw-r--r-- 1 root root  824 Jun 25 06:16 upgrade_metrics_df.csv
-rw-r--r-- 1 root root 5022 Jun 25 06:16 upgrade_run_summary_df.csv
